<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo_CBOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [ ]:
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly

In [ ]:
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [ ]:
pd.options.display.float_format = '{:,.4f}'.format

### ARQUIVO CSV

In [ ]:
# Parametros de entrada
filename = 'quotedata.csv'

# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

### PREPARAÇÃO DO ARQUIVO

In [ ]:
# Isso assume que o formato do arquivo CBOE não foi editado, ou seja, a tabela começa na linha 4
optionsFile = open(filename)
optionsFileData = optionsFile.readlines()
optionsFile.close()

In [ ]:
# Extraindo SPX spot
spotLine = optionsFileData[1]
spotPrice = float(spotLine.split('Last:')[1].split(',')[0])
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

In [ ]:
# Extraindo a data de hoje
dateLine = optionsFileData[2]
todayDate = dateLine.split('Date: ')[1].split(',')
monthDay = todayDate[0].split(' ')

In [ ]:
if len(monthDay) == 2:
    year = int(monthDay[4])
    month = monthDay[2]
    day = int(monthDay[0])
else:
    if monthDay[2].isdigit():
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])
    else:
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])


# criar um dicionário para mapear os nomes dos meses em português para os equivalentes em inglês
nomes_meses = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extrair o nome do mês da string de entrada
nome_mes_pt = monthDay[2]

# converter o nome do mês para inglês usando o dicionário
nome_mes_en = nomes_meses[nome_mes_pt]

# converter o nome do mês para o número correspondente (por exemplo, 'March' -> 3)
num_mes = datetime.strptime(nome_mes_en, '%B').month

# criar o objeto datetime
todayDate = datetime(year=year, month=num_mes, day=day)

In [ ]:
# create a dictionary to map Portuguese month names to English month names
month_names = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extract the month name from the input string
month_name_pt = monthDay[2]

# convert the month name to English using the dictionary
month_name_en = month_names[month_name_pt]

# convert the month name to its corresponding number (e.g., 'March' -> 3)
month_number = datetime.strptime(month_name_en, '%B').month

# create the datetime object
todayDate = datetime(year=year, month=month_number, day=day)

In [ ]:
# Get SPX Options Data
df = pd.read_csv(filename, sep=",", header=None, skiprows=4)
df.columns = ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
              'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
              'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']


df['ExpirationDate'] = pd.to_datetime(df['ExpirationDate'], format='%a %b %d %Y')
df['ExpirationDate'] = df['ExpirationDate'] + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)

### GAMMA - GEX

In [ ]:
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [ ]:
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()

In [ ]:
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.0f}")


# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.0f}")

# Find the positive gamma strike closest to the spot price (Vol Trigger)
positive_gamma_strikes = dfAgg[dfAgg['TotalGamma'] > 0].index
if len(positive_gamma_strikes) > 0:
    # Find the strike closest to the spot price among the positive gamma strikes
    vol_trigger_strike = positive_gamma_strikes[np.abs(positive_gamma_strikes - spotPrice).argmin()]
    vol_trigger_gamma = dfAgg.loc[vol_trigger_strike, 'TotalGamma']
    print(f"\nVol Trigger: {vol_trigger_gamma:.4f} at strike {vol_trigger_strike:.0f}")
else:
    print("\nNo Vol Trigger found (no positive gamma strikes).")


# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

Menores valores de Gamma:
  Put Wall: -3.9918 at strike 6840
  Large Gamma: -3.3020 at strike 6835
  Large Gamma: -1.2017 at strike 6500

Maiores valores de Gamma:
  Call Wall: 25.4129 at strike 6900
  Large Gamma: 22.0864 at strike 6895
  Large Gamma: 11.9928 at strike 7000

Vol Trigger: 22.0864 at strike 6895

Total Gamma: $128.00 Bn per 1% SPX Move


In [ ]:
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [ ]:
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6900, Call GEX: 27.8718 Bn, Put GEX: -2.4589 Bn
  GEX Level 2: Strike 7000, Call GEX: 18.0325 Bn, Put GEX: -6.0398 Bn
  GEX Level 3: Strike 6895, Call GEX: 22.5258 Bn, Put GEX: -0.4394 Bn
  GEX Level 4: Strike 6000, Call GEX: 8.0747 Bn, Put GEX: -9.0939 Bn
  GEX Level 5: Strike 6800, Call GEX: 8.5052 Bn, Put GEX: -5.3069 Bn
  GEX Level 6: Strike 6700, Call GEX: 6.0987 Bn, Put GEX: -6.1000 Bn


In [ ]:
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [ ]:
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [ ]:
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [ ]:
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [ ]:
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (Ex-Next Monthly Expiry): 24.4867 at strike 6825
Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): -3.8827 at strike 6779
Max Gamma Positivo (Ex-Next Monthly Expiry): 92.6634 at strike 6918
Min Gamma Negativo (Ex-Next Monthly Expiry): -65.2063 at strike 5936


In [ ]:
# Consolidating results from CHART 1, CHART 2, and CHART 3

print("--- DADOS CHART 1 ---")
# DADOS CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.0f}")


# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.0f}")

# Find the positive gamma strike closest to the spot price (Vol Trigger) - REMOVED AS REQUESTED

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

print("\n--- DADOS CHART 2 ---")
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

print("\n--- DADOS CHART 3 ---")
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

--- DADOS CHART 1 ---
Menores valores de Gamma:
  Put Wall: -3.9918 at strike 6840
  Large Gamma: -3.3020 at strike 6835
  Large Gamma: -1.2017 at strike 6500

Maiores valores de Gamma:
  Call Wall: 25.4129 at strike 6900
  Large Gamma: 22.0864 at strike 6895
  Large Gamma: 11.9928 at strike 7000

Total Gamma: $128.00 Bn per 1% SPX Move

--- DADOS CHART 2 ---
Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6900, Call GEX: 27.8718 Bn, Put GEX: -2.4589 Bn
  GEX Level 2: Strike 7000, Call GEX: 18.0325 Bn, Put GEX: -6.0398 Bn
  GEX Level 3: Strike 6895, Call GEX: 22.5258 Bn, Put GEX: -0.4394 Bn
  GEX Level 4: Strike 6000, Call GEX: 8.0747 Bn, Put GEX: -9.0939 Bn
  GEX Level 5: Strike 6800, Call GEX: 8.5052 Bn, Put GEX: -5.3069 Bn
  GEX Level 6: Strike 6700, Call GEX: 6.0987 Bn, Put GEX: -6.1000 Bn

--- DADOS CHART 3 ---
Gamma Flip (Ex-Next Monthly Expiry): 24.4867 at strike 6825
Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): -3.8827 a

In [ ]:
# --- Consolidating results for 5 DTE only ---
print("--- DADOS FILTRADOS PARA 5 DTE ---")

# Filter data for 5 DTE
df_5dte = df[df['daysTillExp'] <= 5/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 5 DTE
dfAgg_5dte = df_5dte.groupby(['StrikePrice']).sum(numeric_only=True)

print("\n--- DADOS CHART 1 (5 DTE) ---")
# DADOS CHART 1 (5 DTE)
dfAgg_sorted_5dte = dfAgg_5dte.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values for 5 DTE
smallest_gamma_5dte = dfAgg_sorted_5dte.head(3)
print("Menores valores de Gamma (5 DTE):")
print(f"  Put Wall: {smallest_gamma_5dte.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma_5dte.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma_5dte.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma_5dte.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma_5dte.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma_5dte.index[2]:.0f}")


# Get the 3 largest gamma values for 5 DTE
largest_gamma_5dte = dfAgg_sorted_5dte.tail(3)
print("\nMaiores valores de Gamma (5 DTE):")
print(f"  Call Wall: {largest_gamma_5dte.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma_5dte.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma_5dte.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma_5dte.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma_5dte.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma_5dte.index[0]:.0f}")

# Print the total gamma for 5 DTE
print(f"\nTotal Gamma (5 DTE): ${df_5dte['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

print("\n--- DADOS CHART 2 (5 DTE) ---")
# DADOS CHART 2 (5 DTE)
dfAgg_5dte['AbsoluteTotalGEX'] = dfAgg_5dte['CallGEX'].abs() + dfAgg_5dte['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure for 5 DTE
dfAgg_sorted_gex_5dte = dfAgg_5dte.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX for 5 DTE
gex_levels_5dte = dfAgg_sorted_gex_5dte.head(6)

print("Top 6 GEX Levels (5 DTE - based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels_5dte)):
    strike = gex_levels_5dte.index[i]
    call_gex = gex_levels_5dte.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels_5dte.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

print("\n--- DADOS CHART 3 (5 DTE) ---")
# DADOS CHART 3 (5 DTE)
# To get data for Chart 3 with 5 DTE, we need to re-run the gamma profile calculation with filtered data.
# This part would ideally be a separate function or done earlier in the notebook.
# For now, I will re-filter the original data before calculating the profile.

df_5dte_profile = df[df['daysTillExp'] <= 5/262].copy() # Use a copy to avoid SettingWithCopyWarning

levels_5dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalGamma_5dte = []

# For each spot level, calc gamma exposure at that point for 5 DTE data
for level in levels_5dte:
    df_5dte_profile['callGammaEx'] = df_5dte_profile.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df_5dte_profile['putGammaEx'] = df_5dte_profile.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma_5dte.append(df_5dte_profile['callGammaEx'].sum() - df_5dte_profile['putGammaEx'].sum())

totalGamma_5dte = np.array(totalGamma_5dte) / 10**9

# Find Gamma Flip Point for 5 DTE
zeroCrossIdx_5dte = np.where(np.diff(np.sign(totalGamma_5dte)))[0]

if zeroCrossIdx_5dte.size > 0:
    negGamma_5dte = totalGamma_5dte[zeroCrossIdx_5dte]
    posGamma_5dte = totalGamma_5dte[zeroCrossIdx_5dte+1]
    negStrike_5dte = levels_5dte[zeroCrossIdx_5dte]
    posStrike_5dte = levels_5dte[zeroCrossIdx_5dte+1]

    zeroGamma_5dte = posStrike_5dte - ((posStrike_5dte - negStrike_5dte) * posGamma_5dte/(posGamma_5dte-negGamma_5dte))
    zeroGamma_5dte = zeroGamma_5dte[0] if zeroGamma_5dte.size > 0 else None
else:
    zeroGamma_5dte = None


# Gamma Flip (primeiro valor acima da linha verde central) for 5 DTE
gamma_flip_index_5dte = np.where(totalGamma_5dte > 0)[0][0] if np.any(totalGamma_5dte > 0) else None
if gamma_flip_index_5dte is not None:
    gamma_flip_strike_5dte = levels_5dte[gamma_flip_index_5dte]
    gamma_flip_value_5dte = totalGamma_5dte[gamma_flip_index_5dte]
    print(f"Gamma Flip (5 DTE): {gamma_flip_value_5dte:.4f} at strike {gamma_flip_strike_5dte:.0f}")
else:
    print("No Gamma Flip found (5 DTE).")

# Vol Trigger (value at zero gamma cross) for 5 DTE
if zeroGamma_5dte is not None:
    vol_trigger_value_at_flip_5dte = np.interp(zeroGamma_5dte, levels_5dte, totalGamma_5dte)
    print(f"Vol Trigger (Gamma Flip Point, 5 DTE): {vol_trigger_value_at_flip_5dte:.4f} at strike {zeroGamma_5dte:.0f}")
else:
    print("Vol Trigger not found (5 DTE - no Gamma Flip point).")


# Max Gamma Positivo (5 DTE)
max_gamma_positive_value_5dte = np.max(totalGamma_5dte)
max_gamma_positive_index_5dte = np.argmax(totalGamma_5dte)
max_gamma_positive_strike_5dte = levels_5dte[max_gamma_positive_index_5dte]
print(f"Max Gamma Positivo (5 DTE): {max_gamma_positive_value_5dte:.4f} at strike {max_gamma_positive_strike_5dte:.0f}")


# Min Gamma Negativo (5 DTE)
min_gamma_negative_value_5dte = np.min(totalGamma_5dte)
min_gamma_negative_index_5dte = np.argmin(totalGamma_5dte)
min_gamma_negative_strike_5dte = levels_5dte[min_gamma_negative_index_5dte]
print(f"Min Gamma Negativo (5 DTE): {min_gamma_negative_value_5dte:.4f} at strike {min_gamma_negative_strike_5dte:.0f}")

--- DADOS FILTRADOS PARA 5 DTE ---

--- DADOS CHART 1 (5 DTE) ---
Menores valores de Gamma (5 DTE):
  Put Wall: -4.3569 at strike 6840
  Large Gamma: -3.3470 at strike 6835
  Large Gamma: -0.4469 at strike 6830

Maiores valores de Gamma (5 DTE):
  Call Wall: 21.9436 at strike 6895
  Large Gamma: 19.8719 at strike 6900
  Large Gamma: 6.8525 at strike 6905

Total Gamma (5 DTE): $81.71 Bn per 1% SPX Move

--- DADOS CHART 2 (5 DTE) ---
Top 6 GEX Levels (5 DTE - based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6895, Call GEX: 22.2378 Bn, Put GEX: -0.2942 Bn
  GEX Level 2: Strike 6900, Call GEX: 21.0284 Bn, Put GEX: -1.1565 Bn
  GEX Level 3: Strike 6905, Call GEX: 7.0102 Bn, Put GEX: -0.1577 Bn
  GEX Level 4: Strike 6840, Call GEX: 0.9768 Bn, Put GEX: -5.3337 Bn
  GEX Level 5: Strike 6850, Call GEX: 4.1318 Bn, Put GEX: -1.9717 Bn
  GEX Level 6: Strike 6890, Call GEX: 4.6652 Bn, Put GEX: -0.3776 Bn

--- DADOS CHART 3 (5 DTE) ---
Gamma Flip (5 DTE): 6.1463 at strike

In [ ]:
# --- Consolidating results for 0 DTE only ---
print("--- DADOS FILTRADOS PARA 0 DTE ---")

# Filter data for 0 DTE (using a small epsilon or checking for 0 business days)
# Since daysTillExp is calculated as business days / 262, 0 business days will be 1/262
df_0dte = df[df['daysTillExp'] <= 1/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 0 DTE
dfAgg_0dte = df_0dte.groupby(['StrikePrice']).sum(numeric_only=True)

print("\n--- DADOS CHART 1 (0 DTE) ---")
# DADOS CHART 1 (0 DTE)
dfAgg_sorted_0dte = dfAgg_0dte.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values for 0 DTE
smallest_gamma_0dte = dfAgg_sorted_0dte.head(3)
print("Menores valores de Gamma (0 DTE):")
if not smallest_gamma_0dte.empty:
    print(f"  Put Wall: {smallest_gamma_0dte.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma_0dte.index[0]:.0f}")
    print(f"  Large Gamma: {smallest_gamma_0dte.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma_0dte.index[1]:.0f}")
    print(f"  Large Gamma: {smallest_gamma_0dte.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma_0dte.index[2]:.0f}")
else:
    print("  No data for smallest gamma (0 DTE).")


# Get the 3 largest gamma values for 0 DTE
largest_gamma_0dte = dfAgg_sorted_0dte.tail(3)
print("\nMaiores valores de Gamma (0 DTE):")
if not largest_gamma_0dte.empty:
    print(f"  Call Wall: {largest_gamma_0dte.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma_0dte.index[2]:.0f}")
    print(f"  Large Gamma: {largest_gamma_0dte.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma_0dte.index[1]:.0f}")
    print(f"  Large Gamma: {largest_gamma_0dte.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma_0dte.index[0]:.0f}")
else:
    print("  No data for largest gamma (0 DTE).")

# Print the total gamma for 0 DTE
print(f"\nTotal Gamma (0 DTE): ${df_0dte['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

print("\n--- DADOS CHART 2 (0 DTE) ---")
# DADOS CHART 2 (0 DTE)
dfAgg_0dte['AbsoluteTotalGEX'] = dfAgg_0dte['CallGEX'].abs() + dfAgg_0dte['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure for 0 DTE
dfAgg_sorted_gex_0dte = dfAgg_0dte.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX for 0 DTE
gex_levels_0dte = dfAgg_sorted_gex_0dte.head(6)

print("Top 6 GEX Levels (0 DTE - based on sum of absolute Call and Put Gamma Exposure):")
if not gex_levels_0dte.empty:
    for i in range(len(gex_levels_0dte)):
        strike = gex_levels_0dte.index[i]
        call_gex = gex_levels_0dte.iloc[i]['CallGEX'] / 10**9
        put_gex = gex_levels_0dte.iloc[i]['PutGEX'] / 10**9
        print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")
else:
    print("  No data for Top 6 GEX Levels (0 DTE).")


print("\n--- DADOS CHART 3 (0 DTE) ---")
# DADOS CHART 3 (0 DTE)
# To get data for Chart 3 with 0 DTE, we need to re-run the gamma profile calculation with filtered data.
# This part would ideally be a separate function or done earlier in the notebook.
# For now, I will re-filter the original data before calculating the profile.

df_0dte_profile = df[df['daysTillExp'] <= 1/262].copy() # Use a copy to avoid SettingWithCopyWarning

levels_0dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalGamma_0dte = []

# For each spot level, calc gamma exposure at that point for 0 DTE data
# Need to handle potential division by zero if daysTillExp is exactly 0.
# The calcGammaEx function already has a check for T == 0, which should handle this.
for level in levels_0dte:
    df_0dte_profile['callGammaEx'] = df_0dte_profile.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df_0dte_profile['putGammaEx'] = df_0dte_profile.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma_0dte.append(df_0dte_profile['callGammaEx'].sum() - df_0dte_profile['putGammaEx'].sum())

totalGamma_0dte = np.array(totalGamma_0dte) / 10**9

# Find Gamma Flip Point for 0 DTE
zeroCrossIdx_0dte = np.where(np.diff(np.sign(totalGamma_0dte)))[0]

zeroGamma_0dte = None # Initialize zeroGamma_0dte

if zeroCrossIdx_0dte.size > 0:
    negGamma_0dte = totalGamma_0dte[zeroCrossIdx_0dte]
    posGamma_0dte = totalGamma_0dte[zeroCrossIdx_0dte+1]
    negStrike_0dte = levels_0dte[zeroCrossIdx_0dte]
    posStrike_0dte = levels_0dte[zeroCrossIdx_0dte+1]

    # Handle potential division by zero if negGamma_0dte and posGamma_0dte are equal
    if (posGamma_0dte - negGamma_0dte).all() != 0:
         zeroGamma_0dte = posStrike_0dte - ((posStrike_0dte - negStrike_0dte) * posGamma_0dte/(posGamma_0dte-negGamma_0dte))
         zeroGamma_0dte = zeroGamma_0dte[0] if zeroGamma_0dte.size > 0 else None # Ensure it's a scalar or None
    else:
        zeroGamma_0dte = None # No valid flip point if gamma is constant

# Gamma Flip (primeiro valor acima da linha verde central) for 0 DTE
gamma_flip_index_0dte = np.where(totalGamma_0dte > 0)[0][0] if np.any(totalGamma_0dte > 0) else None
if gamma_flip_index_0dte is not None:
    gamma_flip_strike_0dte = levels_0dte[gamma_flip_index_0dte]
    gamma_flip_value_0dte = totalGamma_0dte[gamma_flip_index_0dte]
    print(f"Gamma Flip (0 DTE): {gamma_flip_value_0dte:.4f} at strike {gamma_flip_strike_0dte:.0f}")
else:
    print("No Gamma Flip found (0 DTE).")

# Vol Trigger (value at zero gamma cross) for 0 DTE
if zeroGamma_0dte is not None:
    vol_trigger_value_at_flip_0dte = np.interp(zeroGamma_0dte, levels_0dte, totalGamma_0dte)
    print(f"Vol Trigger (Gamma Flip Point, 0 DTE): {vol_trigger_value_at_flip_0dte:.4f} at strike {zeroGamma_0dte:.0f}")
else:
    print("Vol Trigger not found (0 DTE - no Gamma Flip point).")


# Max Gamma Positivo (0 DTE)
if totalGamma_0dte.size > 0:
    max_gamma_positive_value_0dte = np.max(totalGamma_0dte)
    max_gamma_positive_index_0dte = np.argmax(totalGamma_0dte)
    max_gamma_positive_strike_0dte = levels_0dte[max_gamma_positive_index_0dte]
    print(f"Max Gamma Positivo (0 DTE): {max_gamma_positive_value_0dte:.4f} at strike {max_gamma_positive_strike_0dte:.0f}")
else:
    print("No data for Max Gamma Positivo (0 DTE).")


# Min Gamma Negativo (0 DTE)
if totalGamma_0dte.size > 0:
    min_gamma_negative_value_0dte = np.min(totalGamma_0dte)
    min_gamma_negative_index_0dte = np.argmin(totalGamma_0dte)
    min_gamma_negative_strike_0dte = levels_0dte[min_gamma_negative_index_0dte]
    print(f"Min Gamma Negativo (0 DTE): {min_gamma_negative_value_0dte:.4f} at strike {min_gamma_negative_strike_0dte:.0f}")
else:
     print("No data for Min Gamma Negativo (0 DTE).")

--- DADOS FILTRADOS PARA 0 DTE ---

--- DADOS CHART 1 (0 DTE) ---
Menores valores de Gamma (0 DTE):
  Put Wall: -4.6987 at strike 6840
  Large Gamma: -3.2631 at strike 6835
  Large Gamma: -0.8713 at strike 6830

Maiores valores de Gamma (0 DTE):
  Call Wall: 21.7284 at strike 6895
  Large Gamma: 17.4021 at strike 6900
  Large Gamma: 6.5594 at strike 6905

Total Gamma (0 DTE): $63.30 Bn per 1% SPX Move

--- DADOS CHART 2 (0 DTE) ---
Top 6 GEX Levels (0 DTE - based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6895, Call GEX: 21.8811 Bn, Put GEX: -0.1527 Bn
  GEX Level 2: Strike 6900, Call GEX: 18.2142 Bn, Put GEX: -0.8121 Bn
  GEX Level 3: Strike 6905, Call GEX: 6.5808 Bn, Put GEX: -0.0214 Bn
  GEX Level 4: Strike 6840, Call GEX: 0.4452 Bn, Put GEX: -5.1439 Bn
  GEX Level 5: Strike 6890, Call GEX: 4.0366 Bn, Put GEX: -0.2389 Bn
  GEX Level 6: Strike 6910, Call GEX: 4.0874 Bn, Put GEX: -0.0259 Bn

--- DADOS CHART 3 (0 DTE) ---
Gamma Flip (0 DTE): 26.2168 at strik

### DELTA - DEX

In [ ]:
# ---=== CALCULATE SPOT DELTA ===---
# Delta Exposure = Unit Delta * Open Interest * Contract Size * Spot Price
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice

# Total Delta considers the sign of delta for calls and puts
df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6 # Converting to millions for better scaling

dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

In [ ]:
# Chart 4: Absolute Delta Exposure
# define os dados
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

# cria um gráfico de barras
fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Delta Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta4.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig_delta4.show()

In [ ]:
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])
chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")
fig_delta5.add_shape(dict(type="line", x0=spotPrice, y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6), x1=spotPrice, y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta5.update_layout(
    width=1750,
    height=800
)

fig_delta5.show()

In [ ]:
# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

In [ ]:
# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [ ]:
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(title=chartTitle_delta6, xaxis_title='Index Price', yaxis_title='Delta Exposure ($ millions/1% move)')
fig_delta6.update_layout(title_text=chartTitle_delta6, title_font=dict(size=20, family="Arial Black"))

fig_delta6.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalDelta),
        x1=spotPrice,
        y1=max(totalDelta),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

# Add the Delta Flip line only if zeroDelta is not None
if zeroDelta is not None:
    fig_delta6.add_shape(
        dict(
            type="line",
            x0=zeroDelta,
            y0=min(totalDelta),
            x1=zeroDelta,
            y1=max(totalDelta),
            line=dict(color="green", width=1.5),
            # Format zeroDelta as a scalar
            name="Delta Flip: " + str("{:,.0f}".format(float(zeroDelta)))
        )
    )


fig_delta6.update_xaxes(range=[fromStrike, toStrike])
fig_delta6.update_yaxes(range=[min(totalDelta), max(totalDelta)])

# Adding shaded areas for positive and negative delta
# Adjust shaded areas to account for potential None zeroDelta
if zeroDelta is not None:
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[min(totalDelta), min(totalDelta), min(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="red",
            opacity=0.1,
            showlegend=False,
            name="Negative Delta"
        )
    )

    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[max(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="green",
            opacity=0.1,
            showlegend=False,
            name="Positive Delta"
        )
    )
else:
     # If no zeroDelta, the entire range is either positive or negative
    fill_color = 'green' if totalDelta[0] >= 0 else 'red'
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, toStrike, toStrike, fromStrike],
            y=[min(totalDelta), min(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor=fill_color,
            opacity=0.1,
            showlegend=False,
            name="Delta Region"
        )
    )


# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta6.update_layout(
    width=1400,
    height=700
)

fig_delta6.show()

In [ ]:
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Find the 3 strikes with the smallest (most negative) total Delta Exposure
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)
print("Menores valores de Delta:")
print(f"  Put Delta Wall: {smallest_delta.iloc[0]['TotalDelta']:.4f} at strike {smallest_delta.index[0]:.0f}")
print(f"  Large Delta: {smallest_delta.iloc[1]['TotalDelta']:.4f} at strike {smallest_delta.index[1]:.0f}")
print(f"  Large Delta: {smallest_delta.iloc[2]['TotalDelta']:.4f} at strike {smallest_delta.index[2]:.0f}")

# Find the 3 strikes with the largest (most positive) total Delta Exposure
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)
print("\nMaiores valores de Delta:")
print(f"  Call Delta Wall: {largest_delta.iloc[2]['TotalDelta']:.4f} at strike {largest_delta.index[2]:.0f}")
print(f"  Large Delta: {largest_delta.iloc[1]['TotalDelta']:.4f} at strike {largest_delta.index[1]:.0f}")
print(f"  Large Delta: {largest_delta.iloc[0]['TotalDelta']:.4f} at strike {largest_delta.index[0]:.0f}")

# Print the total Delta
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts)
print("\n--- DADOS CHART 5 ---")
# Calculate the absolute sum of Call and Put Delta Exposure for each strike
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX
dex_levels = dfAgg_delta_sorted_dex.head(6)

print("Top 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure):")
for i in range(len(dex_levels)):
    strike_dex = dex_levels.index[i]
    call_dex = dex_levels.iloc[i]['CallDEX'] / 10**6
    put_dex = dex_levels.iloc[i]['PutDEX'] / 10**6
    print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")

# DATA FOR CHART 6 (Delta Exposure Profile)
print("\n--- DADOS CHART 6 ---")
# Find Delta Flip (first value above the central green line) for Ex-Next Monthly Expiry Delta Profile
# This assumes totalDeltaExFri is the data for the 'Ex-Next Monthly Expiry' line
delta_flip_index_exfri = np.where(totalDeltaExFri > 0)[0][0] if np.any(totalDeltaExFri > 0) else None
if delta_flip_index_exfri is not None:
    delta_flip_strike_exfri = levels_delta[delta_flip_index_exfri]
    delta_flip_value_exfri = totalDeltaExFri[delta_flip_index_exfri]
    print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_flip_value_exfri:.4f} at strike {delta_flip_strike_exfri:.0f}")
else:
    print("No Delta Flip found (Ex-Next Monthly Expiry).")

# Delta Vol Trigger (value at zero delta cross) for Ex-Next Monthly Expiry
# Use the zeroDelta calculated earlier for the Delta Flip point
if zeroDelta is not None:
    delta_vol_trigger_value_at_flip_exfri = np.interp(zeroDelta, levels_delta, totalDeltaExFri)
    print(f"Delta Vol Trigger (Delta Flip Point, Ex-Next Monthly Expiry): {delta_vol_trigger_value_at_flip_exfri:.4f} at strike {zeroDelta:.0f}")
else:
    print("Delta Vol Trigger not found (no Delta Flip point).")

# Max Positive Delta (Ex-Next Monthly Expiry)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_index_exfri = np.argmax(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[max_delta_positive_index_exfri]
print(f"Max Delta Positivo (Ex-Next Monthly Expiry): {max_delta_positive_value_exfri:.4f} at strike {max_delta_positive_strike_exfri:.0f}")

# Min Negative Delta (Ex-Next Monthly Expiry)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_index_exfri = np.argmin(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[min_delta_negative_index_exfri]
print(f"Min Delta Negativo (Ex-Next Monthly Expiry): {min_delta_negative_value_exfri:.4f} at strike {min_delta_negative_strike_exfri:.0f}")

--- DADOS CHART 4 ---
Menores valores de Delta:
  Put Delta Wall: -3465.9027 at strike 6330
  Large Delta: -1071.9687 at strike 12000
  Large Delta: -606.7589 at strike 5340

Maiores valores de Delta:
  Call Delta Wall: 664707.1167 at strike 5000
  Large Delta: 497535.4317 at strike 6000
  Large Delta: 244821.8495 at strike 4000

Total Delta: $2,925,055.01 Million per 1% SPX Move

--- DADOS CHART 5 ---
Top 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure):
  DEX Level 1: Strike 5000, Call DEX: 697342.8744 Million, Put DEX: -32635.7577 Million
  DEX Level 2: Strike 6000, Call DEX: 576534.9181 Million, Put DEX: -78999.4865 Million
  DEX Level 3: Strike 4000, Call DEX: 251958.1430 Million, Put DEX: -7136.2935 Million
  DEX Level 4: Strike 7000, Call DEX: 92282.2786 Million, Put DEX: -43732.2645 Million
  DEX Level 5: Strike 6700, Call DEX: 87411.9121 Million, Put DEX: -30748.1302 Million
  DEX Level 6: Strike 6600, Call DEX: 82579.3171 Million, Put DEX: -23922.6334 Milli

In [ ]:
# --- Consolidating results for 5 DTE only (Delta) ---
print("--- DADOS FILTRADOS PARA 5 DTE (Delta) ---")

# Filter data for 5 DTE
df_5dte_delta = df[df['daysTillExp'] <= 5/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 5 DTE
dfAgg_5dte_delta = df_5dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalculate Delta Exposure for 5 DTE
dfAgg_5dte_delta['CallDEX'] = dfAgg_5dte_delta['CallDelta'] * dfAgg_5dte_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_5dte_delta['PutDEX'] = dfAgg_5dte_delta['PutDelta'] * dfAgg_5dte_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_5dte_delta['TotalDelta'] = (dfAgg_5dte_delta.CallDEX + dfAgg_5dte_delta.PutDEX) / 10**6 # Converting to millions

print("\n--- DADOS CHART 4 (5 DTE) ---")
# Find the 3 strikes with the smallest (most negative) total Delta Exposure for 5 DTE
smallest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').head(3)
print("Menores valores de Delta (5 DTE):")
if not smallest_delta_5dte.empty:
    print(f"  Put Delta Wall: {smallest_delta_5dte.iloc[0]['TotalDelta']:.4f} at strike {smallest_delta_5dte.index[0]:.0f}")
    print(f"  Large Delta: {smallest_delta_5dte.iloc[1]['TotalDelta']:.4f} at strike {smallest_delta_5dte.index[1]:.0f}")
    print(f"  Large Delta: {smallest_delta_5dte.iloc[2]['TotalDelta']:.4f} at strike {smallest_delta_5dte.index[2]:.0f}")
else:
    print("  No data for smallest delta (5 DTE).")


# Find the 3 strikes with the largest (most positive) total Delta Exposure for 5 DTE
largest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').tail(3)
print("\nMaiores valores de Delta (5 DTE):")
if not largest_delta_5dte.empty:
    print(f"  Call Delta Wall: {largest_delta_5dte.iloc[2]['TotalDelta']:.4f} at strike {largest_delta_5dte.index[2]:.0f}")
    print(f"  Large Delta: {largest_delta_5dte.iloc[1]['TotalDelta']:.4f} at strike {largest_delta_5dte.index[1]:.0f}")
    print(f"  Large Delta: {largest_delta_5dte.iloc[0]['TotalDelta']:.4f} at strike {largest_delta_5dte.index[0]:.0f}")
else:
    print("  No data for largest delta (5 DTE).")

# Print the total Delta for 5 DTE
print(f"\nTotal Delta (5 DTE): ${df_5dte_delta['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts) for 5 DTE
print("\n--- DADOS CHART 5 (5 DTE) ---")
# Calculate the absolute sum of Call and Put Delta Exposure for each strike for 5 DTE
dfAgg_5dte_delta['AbsoluteTotalDEX'] = dfAgg_5dte_delta['CallDEX'].abs() + dfAgg_5dte_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure for 5 DTE
dfAgg_delta_sorted_dex_5dte = dfAgg_5dte_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX for 5 DTE
dex_levels_5dte = dfAgg_delta_sorted_dex_5dte.head(6)

print("Top 6 DEX Levels (5 DTE - based on sum of absolute Call and Put Delta Exposure):")
if not dex_levels_5dte.empty:
    for i in range(len(dex_levels_5dte)):
        strike_dex = dex_levels_5dte.index[i]
        call_dex = dex_levels_5dte.iloc[i]['CallDEX'] / 10**6
        put_dex = dex_levels_5dte.iloc[i]['PutDEX'] / 10**6
        print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")
else:
    print("  No data for Top 6 DEX Levels (5 DTE).")


# DATA FOR CHART 6 (Delta Exposure Profile) for 5 DTE
print("\n--- DADOS CHART 6 (5 DTE) ---")
# To get data for Chart 6 with 5 DTE, we need to re-run the delta profile calculation with filtered data.
# This part would ideally be a separate function or done earlier in the notebook.
# For now, I will re-filter the original data before calculating the profile.

df_5dte_profile_delta = df[df['daysTillExp'] <= 5/262].copy() # Use a copy

levels_delta_5dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalDelta_5dte = []

# For each spot level, calc delta exposure at that point for 5 DTE data
for level in levels_delta_5dte:
    df_5dte_profile_delta['callDeltaEx'] = df_5dte_profile_delta.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df_5dte_profile_delta['putDeltaEx'] = df_5dte_profile_delta.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta_5dte.append(df_5dte_profile_delta['callDeltaEx'].sum() + df_5dte_profile_delta['putDeltaEx'].sum())

totalDelta_5dte = np.array(totalDelta_5dte) / 10**6 # Converting to millions

# Find Delta Flip Point for 5 DTE
zeroCrossIdx_delta_5dte = np.where(np.diff(np.sign(totalDelta_5dte)))[0]

zeroDelta_5dte = None # Initialize zeroDelta_5dte

if zeroCrossIdx_delta_5dte.size > 0:
    negDelta_5dte = totalDelta_5dte[zeroCrossIdx_delta_5dte]
    posDelta_5dte = totalDelta_5dte[zeroCrossIdx_delta_5dte+1]
    negStrike_delta_5dte = levels_delta_5dte[zeroCrossIdx_delta_5dte]
    posStrike_delta_5dte = levels_delta_5dte[zeroCrossIdx_delta_5dte+1]

    if (posDelta_5dte - negDelta_5dte).all() != 0:
         zeroDelta_5dte = posStrike_delta_5dte - ((posStrike_delta_5dte - negStrike_delta_5dte) * posDelta_5dte/(posDelta_5dte-negDelta_5dte))
         zeroDelta_5dte = zeroDelta_5dte[0] if zeroDelta_5dte.size > 0 else None
    else:
        zeroDelta_5dte = None

# Delta Flip (first value above the central green line) for 5 DTE
delta_flip_index_5dte = np.where(totalDelta_5dte > 0)[0][0] if np.any(totalDelta_5dte > 0) else None
if delta_flip_index_5dte is not None:
    delta_flip_strike_5dte = levels_delta_5dte[delta_flip_index_5dte]
    delta_flip_value_5dte = totalDelta_5dte[delta_flip_index_5dte]
    print(f"Delta Flip (5 DTE): {delta_flip_value_5dte:.4f} at strike {delta_flip_strike_5dte:.0f}")
else:
    print("No Delta Flip found (5 DTE).")

# Delta Vol Trigger (value at zero delta cross) for 5 DTE
if zeroDelta_5dte is not None:
    delta_vol_trigger_value_at_flip_5dte = np.interp(zeroDelta_5dte, levels_delta_5dte, totalDelta_5dte)
    print(f"Delta Vol Trigger (Delta Flip Point, 5 DTE): {delta_vol_trigger_value_at_flip_5dte:.4f} at strike {zeroDelta_5dte:.0f}")
else:
    print("Delta Vol Trigger not found (5 DTE - no Delta Flip point).")

# Max Positive Delta (5 DTE)
if totalDelta_5dte.size > 0:
    max_delta_positive_value_5dte = np.max(totalDelta_5dte)
    max_delta_positive_index_5dte = np.argmax(totalDelta_5dte)
    max_delta_positive_strike_5dte = levels_delta_5dte[max_delta_positive_index_5dte]
    print(f"Max Delta Positivo (5 DTE): {max_delta_positive_value_5dte:.4f} at strike {max_delta_positive_strike_5dte:.0f}")
else:
    print("No data for Max Delta Positivo (5 DTE).")

# Min Negative Delta (5 DTE)
if totalDelta_5dte.size > 0:
    min_delta_negative_value_5dte = np.min(totalDelta_5dte)
    min_delta_negative_index_5dte = np.argmin(totalDelta_5dte)
    min_delta_negative_strike_5dte = levels_delta_5dte[min_delta_negative_index_5dte]
    print(f"Min Delta Negativo (5 DTE): {min_delta_negative_value_5dte:.4f} at strike {min_delta_negative_strike_5dte:.0f}")
else:
    print("No data for Min Delta Negativo (5 DTE).")

--- DADOS FILTRADOS PARA 5 DTE (Delta) ---

--- DADOS CHART 4 (5 DTE) ---
Menores valores de Delta (5 DTE):
  Put Delta Wall: -20370.1737 at strike 7000
  Large Delta: -15461.4077 at strike 6835
  Large Delta: -10348.7167 at strike 6840

Maiores valores de Delta (5 DTE):
  Call Delta Wall: 72573.4287 at strike 6900
  Large Delta: 66945.1878 at strike 6895
  Large Delta: 54098.4725 at strike 6750

Total Delta (5 DTE): $182,617.48 Million per 1% SPX Move

--- DADOS CHART 5 (5 DTE) ---
Top 6 DEX Levels (5 DTE - based on sum of absolute Call and Put Delta Exposure):
  DEX Level 1: Strike 6900, Call DEX: 79017.6894 Million, Put DEX: -6444.2608 Million
  DEX Level 2: Strike 6850, Call DEX: 63829.5623 Million, Put DEX: -11136.0962 Million
  DEX Level 3: Strike 6895, Call DEX: 68819.4165 Million, Put DEX: -1874.2287 Million
  DEX Level 4: Strike 6800, Call DEX: 60026.1044 Million, Put DEX: -8383.2439 Million
  DEX Level 5: Strike 6750, Call DEX: 61115.3452 Million, Put DEX: -7016.8727 Million


In [ ]:
# --- Consolidating results for 0 DTE only (Delta) ---
print("--- DADOS FILTRADOS PARA 0 DTE (Delta) ---")

# Filter data for 0 DTE (using a small epsilon or checking for 0 business days)
# Since daysTillExp is calculated as business days / 262, 0 business days will be 1/262
df_0dte_delta = df[df['daysTillExp'] <= 1/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 0 DTE
dfAgg_0dte_delta = df_0dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalculate Delta Exposure for 0 DTE
dfAgg_0dte_delta['CallDEX'] = dfAgg_0dte_delta['CallDelta'] * dfAgg_0dte_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_0dte_delta['PutDEX'] = dfAgg_0dte_delta['PutDelta'] * dfAgg_0dte_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_0dte_delta['TotalDelta'] = (dfAgg_0dte_delta.CallDEX + dfAgg_0dte_delta.PutDEX) / 10**6 # Converting to millions


print("\n--- DADOS CHART 4 (0 DTE) ---")
# Find the 3 strikes with the smallest (most negative) total Delta Exposure for 0 DTE
smallest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').head(3)
print("Menores valores de Delta (0 DTE):")
if not smallest_delta_0dte.empty:
    print(f"  Put Delta Wall: {smallest_delta_0dte.iloc[0]['TotalDelta']:.4f} at strike {smallest_delta_0dte.index[0]:.0f}")
    print(f"  Large Delta: {smallest_delta_0dte.iloc[1]['TotalDelta']:.4f} at strike {smallest_delta_0dte.index[1]:.0f}")
    print(f"  Large Delta: {smallest_delta_0dte.iloc[2]['TotalDelta']:.4f} at strike {smallest_delta_0dte.index[2]:.0f}")
else:
    print("  No data for smallest delta (0 DTE).")


# Find the 3 strikes with the largest (most positive) total Delta Exposure for 0 DTE
largest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').tail(3)
print("\nMaiores valores de Delta (0 DTE):")
if not largest_delta_0dte.empty:
    print(f"  Call Delta Wall: {largest_delta_0dte.iloc[2]['TotalDelta']:.4f} at strike {largest_delta_0dte.index[2]:.0f}")
    print(f"  Large Delta: {largest_delta_0dte.iloc[1]['TotalDelta']:.4f} at strike {largest_delta_0dte.index[1]:.0f}")
    print(f"  Large Delta: {largest_delta_0dte.iloc[0]['TotalDelta']:.4f} at strike {largest_delta_0dte.index[0]:.0f}")
else:
    print("  No data for largest delta (0 DTE).")


# Print the total Delta for 0 DTE
print(f"\nTotal Delta (0 DTE): ${df_0dte_delta['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

print("\n--- DADOS CHART 5 (0 DTE) ---")
# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts) for 0 DTE
dfAgg_0dte_delta['AbsoluteTotalDEX'] = dfAgg_0dte_delta['CallDEX'].abs() + dfAgg_0dte_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure for 0 DTE
dfAgg_delta_sorted_dex_0dte = dfAgg_0dte_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX for 0 DTE
dex_levels_0dte = dfAgg_delta_sorted_dex_0dte.head(6)

print("Top 6 DEX Levels (0 DTE - based on sum of absolute Call and Put Delta Exposure):")
if not dex_levels_0dte.empty:
    for i in range(len(dex_levels_0dte)):
        strike_dex = dex_levels_0dte.index[i]
        call_dex = dex_levels_0dte.iloc[i]['CallDEX'] / 10**6
        put_dex = dex_levels_0dte.iloc[i]['PutDEX'] / 10**6
        print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")
else:
    print("  No data for Top 6 DEX Levels (0 DTE).")


print("\n--- DADOS CHART 6 (0 DTE) ---")
# DATA FOR CHART 6 (Delta Exposure Profile) for 0 DTE
# To get data for Chart 6 with 0 DTE, we need to re-run the delta profile calculation with filtered data.
# This part would ideally be a separate function or done earlier in the notebook.
# For now, I will re-filter the original data before calculating the profile.

df_0dte_profile_delta = df[df['daysTillExp'] <= 1/262].copy() # Use a copy

levels_delta_0dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalDelta_0dte = []

# For each spot level, calc delta exposure at that point for 0 DTE data
for level in levels_delta_0dte:
    df_0dte_profile_delta['callDeltaEx'] = df_0dte_profile_delta.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df_0dte_profile_delta['putDeltaEx'] = df_0dte_profile_delta.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta_0dte.append(df_0dte_profile_delta['callDeltaEx'].sum() + df_0dte_profile_delta['putDeltaEx'].sum())

totalDelta_0dte = np.array(totalDelta_0dte) / 10**6 # Converting to millions


# Find Delta Flip Point for 0 DTE
zeroCrossIdx_delta_0dte = np.where(np.diff(np.sign(totalDelta_0dte)))[0]

zeroDelta_0dte = None # Initialize zeroDelta_0dte

if zeroCrossIdx_delta_0dte.size > 0:
    negDelta_0dte = totalDelta_0dte[zeroCrossIdx_delta_0dte]
    posDelta_0dte = totalDelta_0dte[zeroCrossIdx_delta_0dte+1]
    negStrike_delta_0dte = levels_delta_0dte[zeroCrossIdx_delta_0dte]
    posStrike_delta_0dte = levels_delta_0dte[zeroCrossIdx_delta_0dte+1]

    if (posDelta_0dte - negDelta_0dte).all() != 0:
         zeroDelta_0dte = posStrike_delta_0dte - ((posStrike_delta_0dte - negStrike_delta_0dte) * posDelta_0dte/(posDelta_0dte-negDelta_0dte))
         zeroDelta_0dte = zeroDelta_0dte[0] if zeroDelta_0dte.size > 0 else None
    else:
        zeroDelta_0dte = None

# Delta Flip (first value above the central green line) for 0 DTE
delta_flip_index_0dte = np.where(totalDelta_0dte > 0)[0][0] if np.any(totalDelta_0dte > 0) else None
if delta_flip_index_0dte is not None:
    delta_flip_strike_0dte = levels_delta_0dte[delta_flip_index_0dte]
    delta_flip_value_0dte = totalDelta_0dte[delta_flip_index_0dte]
    print(f"Delta Flip (0 DTE): {delta_flip_value_0dte:.4f} at strike {delta_flip_strike_0dte:.0f}")
else:
    print("No Delta Flip found (0 DTE).")

# Delta Vol Trigger (value at zero delta cross) for 0 DTE
if zeroDelta_0dte is not None:
    delta_vol_trigger_value_at_flip_0dte = np.interp(zeroDelta_0dte, levels_delta_0dte, totalDelta_0dte)
    print(f"Delta Vol Trigger (Delta Flip Point, 0 DTE): {delta_vol_trigger_value_at_flip_0dte:.4f} at strike {zeroDelta_0dte:.0f}")
else:
    print("Delta Vol Trigger not found (0 DTE - no Delta Flip point).")


# Max Positive Delta (0 DTE)
if totalDelta_0dte.size > 0:
    max_delta_positive_value_0dte = np.max(totalDelta_0dte)
    max_delta_positive_index_0dte = np.argmax(totalDelta_0dte)
    max_delta_positive_strike_0dte = levels_delta_0dte[max_delta_positive_index_0dte]
    print(f"Max Delta Positivo (0 DTE): {max_delta_positive_value_0dte:.4f} at strike {max_delta_positive_strike_0dte:.0f}")
else:
    print("No data for Max Delta Positivo (0 DTE).")

# Min Negative Delta (0 DTE)
if totalDelta_0dte.size > 0:
    min_delta_negative_value_0dte = np.min(totalDelta_0dte)
    min_delta_negative_index_0dte = np.argmin(totalDelta_0dte)
    min_delta_negative_strike_0dte = levels_delta_0dte[min_delta_negative_index_0dte]
    print(f"Min Delta Negativo (0 DTE): {min_delta_negative_value_0dte:.4f} at strike {min_delta_negative_strike_0dte:.0f}")
else:
    print("No data for Min Delta Negativo (0 DTE).")

--- DADOS FILTRADOS PARA 0 DTE (Delta) ---

--- DADOS CHART 4 (0 DTE) ---
Menores valores de Delta (0 DTE):
  Put Delta Wall: -2518.9432 at strike 6840
  Large Delta: -2469.3664 at strike 6835
  Large Delta: -25.4534 at strike 7175

Maiores valores de Delta (0 DTE):
  Call Delta Wall: 20439.7175 at strike 6895
  Large Delta: 14164.5790 at strike 6900
  Large Delta: 8122.7588 at strike 6800

Total Delta (0 DTE): $73,162.56 Million per 1% SPX Move

--- DADOS CHART 5 (0 DTE) ---
Top 6 DEX Levels (0 DTE - based on sum of absolute Call and Put Delta Exposure):
  DEX Level 1: Strike 6895, Call DEX: 20596.0751 Million, Put DEX: -156.3576 Million
  DEX Level 2: Strike 6900, Call DEX: 15144.1592 Million, Put DEX: -979.5801 Million
  DEX Level 3: Strike 6850, Call DEX: 8164.6303 Million, Put DEX: -1150.7385 Million
  DEX Level 4: Strike 6800, Call DEX: 8711.0471 Million, Put DEX: -588.2883 Million
  DEX Level 5: Strike 6840, Call DEX: 2576.7579 Million, Put DEX: -5095.7012 Million
  DEX Level 6:

### RESULTADOS GAMMA

In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar UMA ÚNICA LINHA para atualizar tudo de uma vez

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================

# CHART 1 - Spot Gamma Levels
c1_put_wall_value = smallest_gamma.iloc[0]['TotalGamma']
c1_put_wall_strike = smallest_gamma.index[0]
c1_lg1_value = smallest_gamma.iloc[1]['TotalGamma']
c1_lg1_strike = smallest_gamma.index[1]
c1_lg2_value = smallest_gamma.iloc[2]['TotalGamma']
c1_lg2_strike = smallest_gamma.index[2]
c1_call_wall_value = largest_gamma.iloc[2]['TotalGamma']
c1_call_wall_strike = largest_gamma.index[2]
c1_lg3_value = largest_gamma.iloc[1]['TotalGamma']
c1_lg3_strike = largest_gamma.index[1]
c1_lg4_value = largest_gamma.iloc[0]['TotalGamma']
c1_lg4_strike = largest_gamma.index[0]
# c1_vol_trigger_value = vol_trigger_gamma # Removed as requested
# c1_vol_trigger_strike = vol_trigger_strike # Removed as requested

# CHART 2 - GEX Levels
c2_gex1_strike = gex_levels.index[0]
c2_gex1_call = gex_levels.iloc[0]['CallGEX'] / 10**9
c2_gex1_put = gex_levels.iloc[0]['PutGEX'] / 10**9

c2_gex2_strike = gex_levels.index[1]
c2_gex2_call = gex_levels.iloc[1]['CallGEX'] / 10**9
c2_gex2_put = gex_levels.iloc[1]['PutGEX'] / 10**9

c2_gex3_strike = gex_levels.index[2]
c2_gex3_call = gex_levels.iloc[2]['CallGEX'] / 10**9
c2_gex3_put = gex_levels.iloc[2]['PutGEX'] / 10**9

c2_gex4_strike = gex_levels.index[3]
c2_gex4_call = gex_levels.iloc[3]['CallGEX'] / 10**9
c2_gex4_put = gex_levels.iloc[3]['PutGEX'] / 10**9

c2_gex5_strike = gex_levels.index[4]
c2_gex5_call = gex_levels.iloc[4]['CallGEX'] / 10**9
c2_gex5_put = gex_levels.iloc[4]['PutGEX'] / 10**9

c2_gex6_strike = gex_levels.index[5]
c2_gex6_call = gex_levels.iloc[5]['CallGEX'] / 10**9
c2_gex6_put = gex_levels.iloc[5]['PutGEX'] / 10**9

# CHART 3 - Gamma Profile
c3_gamma_flip_value = gamma_flip_value
c3_gamma_flip_strike = gamma_flip_strike
c3_max_pos_value = max_gamma_positive_value
c3_max_pos_strike = max_gamma_positive_strike
c3_min_neg_value = min_gamma_negative_value
c3_min_neg_strike = min_gamma_negative_strike
# Use the interpolated Vol Trigger from Chart 3 data
c3_vol_trigger_value = vol_trigger_value_at_flip_exfri # Use the variable from the Chart 3 data cell
c3_vol_trigger_strike = zeroGamma # Use the zero gamma cross strike


# Dados gerais
spot_price = spotPrice
total_gamma = df['TotalGamma'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA ====================

# Criar a string com todos os dados separados por vírgula
data_string = f"{spot_price},{total_gamma},{update_date},"
data_string += f"{c1_put_wall_value},{c1_put_wall_strike},"
data_string += f"{c1_lg1_value},{c1_lg1_strike},"
data_string += f"{c1_lg2_value},{c1_lg2_strike},"
data_string += f"{c1_call_wall_value},{c1_call_wall_strike},"
data_string += f"{c1_lg3_value},{c1_lg3_strike},"
data_string += f"{c1_lg4_value},{c1_lg4_strike},"
# Replace Chart 1 Vol Trigger with Chart 3 Vol Trigger, keeping the same position
data_string += f"{c3_vol_trigger_value},{c3_vol_trigger_strike},"
data_string += f"{c2_gex1_strike},{c2_gex1_call},{c2_gex1_put},"
data_string += f"{c2_gex2_strike},{c2_gex2_call},{c2_gex2_put},"
data_string += f"{c2_gex3_strike},{c2_gex3_call},{c2_gex3_put},"
data_string += f"{c2_gex4_strike},{c2_gex4_call},{c2_gex4_put},"
data_string += f"{c2_gex5_strike},{c2_gex5_call},{c2_gex5_put},"
data_string += f"{c2_gex6_strike},{c2_gex6_call},{c2_gex6_put},"
data_string += f"{c3_gamma_flip_value},{c3_gamma_flip_strike},"
data_string += f"{c3_max_pos_value},{c3_max_pos_strike},"
data_string += f"{c3_min_neg_value},{c3_min_neg_strike}"

print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS:\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA: ${total_gamma:.2f} Bn")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Levels):")
print(f"   • Put Wall: {c1_put_wall_strike:.0f} (γ: {c1_put_wall_value:.2f})")
print(f"   • Large 1: {c1_lg1_strike:.0f} (γ: {c1_lg1_value:.2f})")
print(f"   • Large 2: {c1_lg2_strike:.0f} (γ: {c1_lg2_value:.2f})")

print("\n🟢 RESISTÊNCIAS (Call Levels):")
print(f"   • Call Wall: {c1_call_wall_strike:.0f} (γ: {c1_call_wall_value:.2f})")
print(f"   • Large 3: {c1_lg3_strike:.0f} (γ: {c1_lg3_value:.2f})")
print(f"   • Large 4: {c1_lg4_strike:.0f} (γ: {c1_lg4_value:.2f})")

print("\n🟠 NÍVEIS ESPECIAIS:")
print(f"   • Vol Trigger: {c3_vol_trigger_strike:.0f}") # Use Chart 3 Vol Trigger strike for summary
print(f"   • Gamma Flip: {c3_gamma_flip_strike:.0f}")

print("\n💎 TOP 3 GEX LEVELS:")
print(f"   1. Strike {c2_gex1_strike:.0f}: Net GEX = {(c2_gex1_call + c2_gex1_put):.2f} Bn")
print(f"   2. Strike {c2_gex2_strike:.0f}: Net GEX = {(c2_gex2_call + c2_gex2_put):.2f} Bn")
print(f"   3. Strike {c2_gex3_strike:.0f}: Net GEX = {(c2_gex3_call + c2_gex3_put):.2f} Bn")

regime = "POSITIVE GAMMA ✅" if spot_price > c3_gamma_flip_strike else "NEGATIVE GAMMA ⚠️"
print(f"\n📈 REGIME: {regime}")

distance_to_flip = ((spot_price - c3_gamma_flip_strike) / c3_gamma_flip_strike) * 100
print(f"📏 DISTÂNCIA DO FLIP: {distance_to_flip:.2f}%")

print("\n" + "="*80)
print("✅ DADOS PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO ====================

print("🔍 VALIDAÇÃO:\n")

errors = []
if spot_price <= 0:
    errors.append("❌ Spot Price inválido")
if c1_put_wall_strike <= 0:
    errors.append("❌ Put Wall inválido")
if c1_call_wall_strike <= 0:
    errors.append("❌ Call Wall inválido")
if c3_gamma_flip_strike <= 0:
    errors.append("❌ Gamma Flip inválido")
if c3_vol_trigger_strike <= 0: # Add validation for the new Vol Trigger strike
     errors.append("❌ Vol Trigger inválido")


if errors:
    print("⚠️ AVISOS:")
    for error in errors:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados!")
    print("✅ String pronta para uso!")

print("\n" + "="*80 + "\n")
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar o código TradingView automaticamente

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================

# CHART 1 - Spot Gamma Levels
chart1_data = {
    'Put Wall': {
        'value': smallest_gamma.iloc[0]['TotalGamma'],
        'strike': smallest_gamma.index[0]
    },
    'Large Gamma 1': {
        'value': smallest_gamma.iloc[1]['TotalGamma'],
        'strike': smallest_gamma.index[1]
    },
    'Large Gamma 2': {
        'value': smallest_gamma.iloc[2]['TotalGamma'],
        'strike': smallest_gamma.index[2]
    },
    'Call Wall': {
        'value': largest_gamma.iloc[2]['TotalGamma'],
        'strike': largest_gamma.index[2]
    },
    'Large Gamma 3': {
        'value': largest_gamma.iloc[1]['TotalGamma'],
        'strike': largest_gamma.index[1]
    },
    'Large Gamma 4': {
        'value': largest_gamma.iloc[0]['TotalGamma'],
        'strike': largest_gamma.index[0]
    },
    # 'Vol Trigger': { # Removed as requested
    #     'value': vol_trigger_gamma,
    #     'strike': vol_trigger_strike
    # }
}

# CHART 2 - GEX Levels
chart2_data = {}
for i in range(min(6, len(gex_levels))):
    chart2_data[f'GEX Level {i+1}'] = {
        'strike': gex_levels.index[i],
        'call_gex': gex_levels.iloc[i]['CallGEX'] / 10**9,
        'put_gex': gex_levels.iloc[i]['PutGEX'] / 10**9
    }

# CHART 3 - Gamma Profile
chart3_data = {
    'Gamma Flip': {
        'value': gamma_flip_value,
        'strike': gamma_flip_strike
    },
    'Max Gamma Positivo': {
        'value': max_gamma_positive_value,
        'strike': max_gamma_positive_strike
    },
    'Min Gamma Negativo': {
        'value': min_gamma_negative_value,
        'strike': min_gamma_negative_strike
    },
    'Vol Trigger': { # Add Vol Trigger from Chart 3 data
        'value': vol_trigger_value_at_flip_exfri,
        'strike': zeroGamma
    }
}

# Dados gerais
general_data = {
    'spot_price': spotPrice,
    'total_gamma': df['TotalGamma'].sum(),
    'update_date': todayDate.strftime('%d %b %Y 00:00')
}

# ==================== GERAÇÃO DO CÓDIGO ====================

print("📋 CÓDIGO PARA TRADINGVIEW - COPIE E COLE NOS INPUTS\n")
print("="*80)
print("// Cole este código no indicador TradingView")
print("// Substitua apenas os VALORES dos inputs existentes")
print("="*80 + "\n")

# DADOS GERAIS
print("// ==================== DADOS GERAIS ====================")
print(f"spotPrice = {general_data['spot_price']:.2f}")
print(f"totalGamma = {general_data['total_gamma']:.2f}")
print(f'updateDate = timestamp("{general_data["update_date"]}")')
print()

# CHART 1
print("// ==================== CHART 1: SPOT GAMMA LEVELS ====================")
print(f"c1_put_wall = {chart1_data['Put Wall']['value']:.4f}")
print(f"c1_put_wall_strike = {chart1_data['Put Wall']['strike']:.0f}")
print()
print(f"c1_lg1 = {chart1_data['Large Gamma 1']['value']:.4f}")
print(f"c1_lg1_strike = {chart1_data['Large Gamma 1']['strike']:.0f}")
print()
print(f"c1_lg2 = {chart1_data['Large Gamma 2']['value']:.4f}")
print(f"c1_lg2_strike = {chart1_data['Large Gamma 2']['strike']:.0f}")
print()
print(f"c1_call_wall = {chart1_data['Call Wall']['value']:.4f}")
print(f"c1_call_wall_strike = {chart1_data['Call Wall']['strike']:.0f}")
print()
print(f"c1_lg3 = {chart1_data['Large Gamma 3']['value']:.4f}")
print(f"c1_lg3_strike = {chart1_data['Large Gamma 3']['strike']:.0f}")
print()
print(f"c1_lg4 = {chart1_data['Large Gamma 4']['value']:.4f}")
print(f"c1_lg4_strike = {chart1_data['Large Gamma 4']['strike']:.0f}")
print()
# Print the Vol Trigger data from Chart 3 here to maintain order in TradingView inputs
print(f"c1_vol_trigger = {chart3_data['Vol Trigger']['value']:.4f}")
print(f"c1_vol_trigger_strike = {chart3_data['Vol Trigger']['strike']:.0f}")
print()


# CHART 2
print("// ==================== CHART 2: GEX LEVELS ====================")
for key, value in chart2_data.items():
    idx = key.split()[-1]  # Pega o número
    print(f"c2_gex{idx}_strike = {value['strike']:.0f}")
    print(f"c2_gex{idx}_call = {value['call_gex']:.4f}")
    print(f"c2_gex{idx}_put = {value['put_gex']:.4f}")
    print()

# CHART 3
print("// ==================== CHART 3: GAMMA PROFILE ====================")
print(f"c3_gamma_flip = {chart3_data['Gamma Flip']['value']:.4f}")
print(f"c3_gamma_flip_strike = {chart3_data['Gamma Flip']['strike']:.0f}")
print()
print(f"c3_max_pos = {chart3_data['Max Gamma Positivo']['value']:.4f}")
print(f"c3_max_pos_strike = {chart3_data['Max Gamma Positivo']['strike']:.0f}")
print()
print(f"c3_min_neg = {chart3_data['Min Gamma Negativo']['value']:.4f}")
print(f"c3_min_neg_strike = {chart3_data['Min Gamma Negativo']['strike']:.0f}")
# Removed the Vol Trigger from here as it's now printed in the Chart 1 section for ordering
# print(f"c3_vol_trigger = {chart3_data['Vol Trigger']['value']:.4f}")
# print(f"c3_vol_trigger_strike = {chart3_data['Vol Trigger']['strike']:.0f}")
print()


print("\n" + "="*80)
print("✅ CÓDIGO GERADO COM SUCESSO!")
print("="*80 + "\n")

# ==================== RESUMO DOS DADOS ====================

print("📊 RESUMO DOS NÍVEIS IMPORTANTES:\n")

print("🔴 SUPORTES (Put Levels):")
print(f"   • Put Wall: {chart1_data['Put Wall']['strike']:.0f} (Gamma: {chart1_data['Put Wall']['value']:.2f})")
print(f"   • Large 1: {chart1_data['Large Gamma 1']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 1']['value']:.2f})")
print(f"   • Large 2: {chart1_data['Large Gamma 2']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 2']['value']:.2f})")

print("\n🟢 RESISTÊNCIAS (Call Levels):")
print(f"   • Call Wall: {chart1_data['Call Wall']['strike']:.0f} (Gamma: {chart1_data['Call Wall']['value']:.2f})")
print(f"   • Large 3: {chart1_data['Large Gamma 3']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 3']['value']:.2f})")
print(f"   • Large 4: {chart1_data['Large Gamma 4']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 4']['value']:.2f})")

print("\n🟠 NÍVEIS ESPECIAIS:")
print(f"   • Vol Trigger: {chart3_data['Vol Trigger']['strike']:.0f}") # Use Chart 3 Vol Trigger strike for summary
print(f"   • Gamma Flip: {chart3_data['Gamma Flip']['strike']:.0f}")

print("\n💎 TOP 3 GEX LEVELS:")
for i in range(min(3, len(chart2_data))):
    key = f'GEX Level {i+1}'
    data = chart2_data[key]
    net_gex = data['call_gex'] + data['put_gex']
    print(f"   {i+1}. Strike {data['strike']:.0f}: Net GEX = {net_gex:.2f} Bn")

print("\n📈 STATUS DO MERCADO:")
print(f"   • Spot Price: ${general_data['spot_price']:,.2f}")
print(f"   • Total Gamma: ${general_data['total_gamma']:.2f} Bn")

regime = "POSITIVE GAMMA ✅" if general_data['spot_price'] > chart3_data['Gamma Flip']['strike'] else "NEGATIVE GAMMA ⚠️"
print(f"   • Regime: {regime}")

distance_to_flip = ((general_data['spot_price'] - chart3_data['Gamma Flip']['strike']) /
                    chart3_data['Gamma Flip']['strike']) * 100
print(f"   • Distância do Flip: {distance_to_flip:.2f}%")

print(f"\n🕐 Última atualização: {general_data['update_date']}")

print("\n" + "="*80)
print("💡 PRÓXIMOS PASSOS:")
print("="*80)
print("1. ✅ Copie os valores acima")
print("2. ✅ Abra o indicador no TradingView")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Cole os valores nos inputs correspondentes")
print("5. ✅ Clique em 'OK' para aplicar")
print("\n💾 Dica: Salve este código em um arquivo .txt para referência futura!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO ====================

print("🔍 VALIDAÇÃO DOS DADOS:\n")

# Verifica se há dados válidos
errors = []

if spotPrice <= 0:
    errors.append("❌ Spot Price inválido")

if not any([v['strike'] > 0 for v in chart1_data.values()]):
    errors.append("❌ Strikes do Chart 1 inválidos")

if not any([v['strike'] > 0 for v in chart2_data.values()]):
    errors.append("❌ Strikes do Chart 2 inválidos")

if not any([v['strike'] > 0 for v in chart3_data.values()]):
    errors.append("❌ Strikes do Chart 3 inválidos")

if chart3_data['Vol Trigger']['strike'] <= 0: # Validate the Vol Trigger strike from Chart 3 data
    errors.append("❌ Vol Trigger inválido")


if errors:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

6894.6499,128.0037844024832,28 Oct 2025 00:00,-3.991814134570997,6840.0,-3.3019735937920336,6835.0,-1.2017483416555208,6500.0,25.412870060891425,6900.0,22.086391557420335,6895.0,11.992774101227996,7000.0,-3.882656281293791,6779.136070119109,6900.0,27.871755921230676,-2.4588858603392487,7000.0,18.032542685446476,-6.03976858421848,6895.0,22.525792396641272,-0.4394008392209395,6000.0,8.074693362201502,-9.093883691962816,6800.0,8.505195425298444,-5.306893524074912,6700.0,6.098722976039956,-6.100020714224706,24.486675082755614,6824.534816271186,92.66338347919778,6918.021594576271,-65.20632874843722,5936.410422372882

💡 COMO USAR:

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'
5. ✅ COLE a linha 

In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK (5 DTE) ====================
# Este célula consolida os resultados filtrados para 5 DTE e gera o código TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (5 DTE - VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (5 DTE) ====================

# CHART 1 - Spot Gamma Levels (5 DTE)
c1_put_wall_value_5dte = smallest_gamma_5dte.iloc[0]['TotalGamma']
c1_put_wall_strike_5dte = smallest_gamma_5dte.index[0]
c1_lg1_value_5dte = smallest_gamma_5dte.iloc[1]['TotalGamma']
c1_lg1_strike_5dte = smallest_gamma_5dte.index[1]
c1_lg2_value_5dte = smallest_gamma_5dte.iloc[2]['TotalGamma']
c1_lg2_strike_5dte = smallest_gamma_5dte.index[2]
c1_call_wall_value_5dte = largest_gamma_5dte.iloc[2]['TotalGamma']
c1_call_wall_strike_5dte = largest_gamma_5dte.index[2]
c1_lg3_value_5dte = largest_gamma_5dte.iloc[1]['TotalGamma']
c1_lg3_strike_5dte = largest_gamma_5dte.index[1]
c1_lg4_value_5dte = largest_gamma_5dte.iloc[0]['TotalGamma']
c1_lg4_strike_5dte = largest_gamma_5dte.index[0]
# Use the interpolated Vol Trigger from Chart 3 data for 5 DTE
c3_vol_trigger_value_5dte = vol_trigger_value_at_flip_5dte # Use the variable from the Chart 3 data cell for 5 DTE
c3_vol_trigger_strike_5dte = zeroGamma_5dte # Use the zero gamma cross strike for 5 DTE


# CHART 2 - GEX Levels (5 DTE)
c2_gex1_strike_5dte = gex_levels_5dte.index[0]
c2_gex1_call_5dte = gex_levels_5dte.iloc[0]['CallGEX'] / 10**9
c2_gex1_put_5dte = gex_levels_5dte.iloc[0]['PutGEX'] / 10**9

c2_gex2_strike_5dte = gex_levels_5dte.index[1]
c2_gex2_call_5dte = gex_levels_5dte.iloc[1]['CallGEX'] / 10**9
c2_gex2_put_5dte = gex_levels_5dte.iloc[1]['PutGEX'] / 10**9

c2_gex3_strike_5dte = gex_levels_5dte.index[2]
c2_gex3_call_5dte = gex_levels_5dte.iloc[2]['CallGEX'] / 10**9
c2_gex3_put_5dte = gex_levels_5dte.iloc[2]['PutGEX'] / 10**9

c2_gex4_strike_5dte = gex_levels_5dte.index[3]
c2_gex4_call_5dte = gex_levels_5dte.iloc[3]['CallGEX'] / 10**9
c2_gex4_put_5dte = gex_levels_5dte.iloc[3]['PutGEX'] / 10**9

c2_gex5_strike_5dte = gex_levels_5dte.index[4]
c2_gex5_call_5dte = gex_levels_5dte.iloc[4]['CallGEX'] / 10**9
c2_gex5_put_5dte = gex_levels_5dte.iloc[4]['PutGEX'] / 10**9

c2_gex6_strike_5dte = gex_levels_5dte.index[5]
c2_gex6_call_5dte = gex_levels_5dte.iloc[5]['CallGEX'] / 10**9
c2_gex6_put_5dte = gex_levels_5dte.iloc[5]['PutGEX'] / 10**9


# CHART 3 - Gamma Profile (5 DTE)
c3_gamma_flip_value_5dte = gamma_flip_value_5dte
c3_gamma_flip_strike_5dte = gamma_flip_strike_5dte
c3_max_pos_value_5dte = max_gamma_positive_value_5dte
c3_max_pos_strike_5dte = max_gamma_positive_strike_5dte
c3_min_neg_value_5dte = min_gamma_negative_value_5dte
c3_min_neg_strike_5dte = min_gamma_negative_strike_5dte


# Dados gerais (using 5 DTE total gamma)
spot_price = spotPrice # Spot price remains the same
total_gamma_5dte = df_5dte['TotalGamma'].sum() # Use total gamma for 5 DTE
update_date = todayDate.strftime('%d %b %Y 00:00') # Date remains the same

# ==================== GERAÇÃO DA LINHA ÚNICA (5 DTE) ====================

# Criar a string com todos os dados separados por vírgula
data_string_5dte = f"{spot_price},{total_gamma_5dte},{update_date},"
data_string_5dte += f"{c1_put_wall_value_5dte},{c1_put_wall_strike_5dte},"
data_string_5dte += f"{c1_lg1_value_5dte},{c1_lg1_strike_5dte},"
data_string_5dte += f"{c1_lg2_value_5dte},{c1_lg2_strike_5dte},"
data_string_5dte += f"{c1_call_wall_value_5dte},{c1_call_wall_strike_5dte},"
data_string_5dte += f"{c1_lg3_value_5dte},{c1_lg3_strike_5dte},"
data_string_5dte += f"{c1_lg4_value_5dte},{c1_lg4_strike_5dte},"
data_string_5dte += f"{c3_vol_trigger_value_5dte},{c3_vol_trigger_strike_5dte}," # Use Chart 3 Vol Trigger (5 DTE)
data_string_5dte += f"{c2_gex1_strike_5dte},{c2_gex1_call_5dte},{c2_gex1_put_5dte},"
data_string_5dte += f"{c2_gex2_strike_5dte},{c2_gex2_call_5dte},{c2_gex2_put_5dte},"
data_string_5dte += f"{c2_gex3_strike_5dte},{c2_gex3_call_5dte},{c2_gex3_put_5dte},"
data_string_5dte += f"{c2_gex4_strike_5dte},{c2_gex4_call_5dte},{c2_gex4_put_5dte},"
data_string_5dte += f"{c2_gex5_strike_5dte},{c2_gex5_call_5dte},{c2_gex5_put_5dte},"
data_string_5dte += f"{c2_gex6_strike_5dte},{c2_gex6_call_5dte},{c2_gex6_put_5dte},"
data_string_5dte += f"{c3_gamma_flip_value_5dte},{c3_gamma_flip_strike_5dte},"
data_string_5dte += f"{c3_max_pos_value_5dte},{c3_max_pos_strike_5dte},"
data_string_5dte += f"{c3_min_neg_value_5dte},{c3_min_neg_strike_5dte}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (5 DTE):\n")
print("="*80)
print(data_string_5dte)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR (5 DTE):\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (5 DTE):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA (5 DTE): ${total_gamma_5dte:.2f} Bn")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Levels - 5 DTE):")
if not smallest_gamma_5dte.empty:
    print(f"   • Put Wall: {smallest_gamma_5dte.index[0]:.0f} (γ: {smallest_gamma_5dte.iloc[0]['TotalGamma']:.2f})")
    print(f"   • Large 1: {smallest_gamma_5dte.index[1]:.0f} (γ: {smallest_gamma_5dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 2: {smallest_gamma_5dte.index[2]:.0f} (γ: {smallest_gamma_5dte.iloc[2]['TotalGamma']:.2f})")
else:
    print("   • No data for Put Levels (5 DTE)")

print("\n🟢 RESISTÊNCIAS (Call Levels - 5 DTE):")
if not largest_gamma_5dte.empty:
    print(f"   • Call Wall: {largest_gamma_5dte.index[2]:.0f} (γ: {largest_gamma_5dte.iloc[2]['TotalGamma']:.2f})")
    print(f"   • Large 3: {largest_gamma_5dte.index[1]:.0f} (γ: {largest_gamma_5dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 4: {largest_gamma_5dte.index[0]:.0f} (γ: {largest_gamma_5dte.iloc[0]['TotalGamma']:.2f})")
else:
    print("   • No data for Call Levels (5 DTE)")


print("\n🟠 NÍVEIS ESPECIAIS (5 DTE):")
if c3_vol_trigger_strike_5dte is not None:
    print(f"   • Vol Trigger: {c3_vol_trigger_strike_5dte:.0f}")
else:
    print("   • Vol Trigger not found (5 DTE)")

if c3_gamma_flip_strike_5dte is not None:
    print(f"   • Gamma Flip: {c3_gamma_flip_strike_5dte:.0f}")
else:
    print("   • Gamma Flip not found (5 DTE)")


print("\n💎 TOP 3 GEX LEVELS (5 DTE):")
if not gex_levels_5dte.empty:
    for i in range(min(3, len(gex_levels_5dte))):
        data = gex_levels_5dte.iloc[i]
        net_gex = (data['CallGEX'] + data['PutGEX']) / 10**9
        print(f"   {i+1}. Strike {data.name:.0f}: Net GEX = {net_gex:.2f} Bn")
else:
    print("   • No data for Top 3 GEX Levels (5 DTE)")


print("\n📈 STATUS DO MERCADO (5 DTE):")
print(f"   • Spot Price: ${spot_price:,.2f}")
print(f"   • Total Gamma (5 DTE): ${total_gamma_5dte:.2f} Bn")

regime_5dte = "POSITIVE GAMMA ✅" if spot_price > (c3_gamma_flip_strike_5dte if c3_gamma_flip_strike_5dte is not None else float('inf')) else "NEGATIVE GAMMA ⚠️"
print(f"   • Regime (5 DTE): {regime_5dte}")

if c3_gamma_flip_strike_5dte is not None and c3_gamma_flip_strike_5dte != 0:
    distance_to_flip_5dte = ((spot_price - c3_gamma_flip_strike_5dte) / c3_gamma_flip_strike_5dte) * 100
    print(f"   • Distância do Flip (5 DTE): {distance_to_flip_5dte:.2f}%")
else:
     print("   • Distância do Flip (5 DTE): N/A (Gamma Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("💡 PRÓXIMOS PASSOS:")
print("="*80)
print("1. ✅ Copie os valores acima")
print("2. ✅ Abra o indicador no TradingView")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK' para aplicar")
print("\n💾 Dica: Salve este código em um arquivo .txt para referência futura!")
print("="*80 + "\n")


# ==================== VALIDAÇÃO FINAL (5 DTE) ====================

print("🔍 VALIDAÇÃO FINAL (5 DTE):\n")

# Verifica se há dados válidos
errors_final_5dte = []

if spotPrice <= 0:
    errors_final_5dte.append("❌ Spot Price inválido")

if smallest_gamma_5dte.empty and largest_gamma_5dte.empty and gex_levels_5dte.empty and (c3_gamma_flip_strike_5dte is None or c3_vol_trigger_strike_5dte is None):
    errors_final_5dte.append("❌ Nenhum dado válido encontrado para 5 DTE.")


if errors_final_5dte:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors_final_5dte:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso (5 DTE)!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


# ==================== CÓDIGO TRADINGVIEW (5 DTE) ====================

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (5 DTE)")
print("="*80 + "\n")

print("📋 CÓDIGO PARA TRADINGVIEW (5 DTE) - COPIE E COLE NOS INPUTS\n")
print("="*80)
print("// Cole este código no indicador TradingView (para 5 DTE)")
print("// Substitua apenas os VALORES dos inputs existentes")
print("="*80 + "\n")

# DADOS GERAIS (5 DTE)
print("// ==================== DADOS GERAIS (5 DTE) ====================")
print(f"spotPrice = {spot_price:.2f}")
print(f"totalGamma = {total_gamma_5dte:.2f}")
print(f'updateDate = timestamp("{update_date}")')
print()

# CHART 1 (5 DTE)
print("// ==================== CHART 1: SPOT GAMMA LEVELS (5 DTE) ====================")
if not smallest_gamma_5dte.empty:
    print(f"c1_put_wall = {c1_put_wall_value_5dte:.4f}")
    print(f"c1_put_wall_strike = {c1_put_wall_strike_5dte:.0f}")
    print()
    print(f"c1_lg1 = {c1_lg1_value_5dte:.4f}")
    print(f"c1_lg1_strike = {c1_lg1_strike_5dte:.0f}")
    print()
    print(f"c1_lg2 = {c1_lg2_value_5dte:.4f}")
    print(f"c1_lg2_strike = {c1_lg2_strike_5dte:.0f}")
    print()
else:
    print("// No data for smallest gamma (5 DTE)")
    print(f"c1_put_wall = 0.0")
    print(f"c1_put_wall_strike = 0")
    print()
    print(f"c1_lg1 = 0.0")
    print(f"c1_lg1_strike = 0")
    print()
    print(f"c1_lg2 = 0.0")
    print(f"c1_lg2_strike = 0")
    print()

if not largest_gamma_5dte.empty:
    print(f"c1_call_wall = {c1_call_wall_value_5dte:.4f}")
    print(f"c1_call_wall_strike = {c1_call_wall_strike_5dte:.0f}")
    print()
    print(f"c1_lg3 = {c1_lg3_value_5dte:.4f}")
    print(f"c1_lg3_strike = {c1_lg3_strike_5dte:.0f}")
    print()
    print(f"c1_lg4 = {c1_lg4_value_5dte:.4f}")
    print(f"c1_lg4_strike = {c1_lg4_strike_5dte:.0f}")
    print()
else:
    print("// No data for largest gamma (5 DTE)")
    print(f"c1_call_wall = 0.0")
    print(f"c1_call_wall_strike = 0")
    print()
    print(f"c1_lg3 = 0.0")
    print(f"c1_lg3_strike = 0")
    print()
    print(f"c1_lg4 = 0.0")
    print(f"c1_lg4_strike = 0")
    print()


# Print the Vol Trigger data from Chart 3 here to maintain order in TradingView inputs (5 DTE)
if c3_vol_trigger_strike_5dte is not None:
    print(f"c1_vol_trigger = {c3_vol_trigger_value_5dte:.4f}")
    print(f"c1_vol_trigger_strike = {c3_vol_trigger_strike_5dte:.0f}")
else:
    print("// Vol Trigger not found (5 DTE)")
    print(f"c1_vol_trigger = 0.0")
    print(f"c1_vol_trigger_strike = 0")
print()


# CHART 2 (5 DTE)
print("// ==================== CHART 2: GEX LEVELS (5 DTE) ====================")
if not gex_levels_5dte.empty:
    for i in range(min(6, len(gex_levels_5dte))):
        key = f'GEX Level {i+1}'
        data = gex_levels_5dte.iloc[i]
        idx = i + 1  # Use index + 1 for naming
        print(f"c2_gex{idx}_strike = {data.name:.0f}")
        print(f"c2_gex{idx}_call = {data['CallGEX'] / 10**9:.4f}")
        print(f"c2_gex{idx}_put = {data['PutGEX'] / 10**9:.4f}")
        print()
else:
    print("// No data for GEX Levels (5 DTE)")
    for i in range(6):
        idx = i + 1
        print(f"c2_gex{idx}_strike = 0")
        print(f"c2_gex{idx}_call = 0.0")
        print(f"c2_gex{idx}_put = 0.0")
        print()


# CHART 3 (5 DTE)
print("// ==================== CHART 3: GAMMA PROFILE (5 DTE) ====================")
if c3_gamma_flip_strike_5dte is not None:
    print(f"c3_gamma_flip = {c3_gamma_flip_value_5dte:.4f}")
    print(f"c3_gamma_flip_strike = {c3_gamma_flip_strike_5dte:.0f}")
    print()
else:
    print("// Gamma Flip not found (5 DTE)")
    print(f"c3_gamma_flip = 0.0")
    print(f"c3_gamma_flip_strike = 0")
    print()

# Max Gamma Positivo (5 DTE)
if 'max_gamma_positive_value_5dte' in locals() and 'max_gamma_positive_strike_5dte' in locals():
     print(f"c3_max_pos = {max_gamma_positive_value_5dte:.4f}")
     print(f"c3_max_pos_strike = {max_gamma_positive_strike_5dte:.0f}")
     print()
else:
     print("// Max Gamma Positivo not found (5 DTE)")
     print(f"c3_max_pos = 0.0")
     print(f"c3_max_pos_strike = 0")
     print()

# Min Gamma Negativo (5 DTE)
if 'min_gamma_negative_value_5dte' in locals() and 'min_gamma_negative_strike_5dte' in locals():
    print(f"c3_min_neg = {min_gamma_negative_value_5dte:.4f}")
    print(f"c3_min_neg_strike = {min_gamma_negative_strike_5dte:.0f}")
    print()
else:
    print("// Min Gamma Negativo not found (5 DTE)")
    print(f"c3_min_neg = 0.0")
    print(f"c3_min_neg_strike = 0")
    print()


print("\n" + "="*80)
print("✅ CÓDIGO PARA 5 DTE GERADO COM SUCESSO!")
print("="*80 + "\n")


# ==================== RESUMO DOS DADOS (5 DTE) ====================

print("📊 RESUMO DOS NÍVEIS IMPORTANTES (5 DTE):\n")

print("🔴 SUPORTES (Put Levels - 5 DTE):")
if not smallest_gamma_5dte.empty:
    print(f"   • Put Wall: {smallest_gamma_5dte.index[0]:.0f} (Gamma: {smallest_gamma_5dte.iloc[0]['TotalGamma']:.2f})")
    print(f"   • Large 1: {smallest_gamma_5dte.index[1]:.0f} (Gamma: {smallest_gamma_5dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 2: {smallest_gamma_5dte.index[2]:.0f} (Gamma: {smallest_gamma_5dte.iloc[2]['TotalGamma']:.2f})")
else:
    print("   • No data for Put Levels (5 DTE)")

print("\n🟢 RESISTÊNCIAS (Call Levels - 5 DTE):")
if not largest_gamma_5dte.empty:
    print(f"   • Call Wall: {largest_gamma_5dte.index[2]:.0f} (Gamma: {largest_gamma_5dte.iloc[2]['TotalGamma']:.2f})")
    print(f"   • Large 3: {largest_gamma_5dte.index[1]:.0f} (Gamma: {largest_gamma_5dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 4: {largest_gamma_5dte.index[0]:.0f} (Gamma: {largest_gamma_5dte.iloc[0]['TotalGamma']:.2f})")
else:
    print("   • No data for Call Levels (5 DTE)")


print("\n🟠 NÍVEIS ESPECIAIS (5 DTE):")
if c3_vol_trigger_strike_5dte is not None:
    print(f"   • Vol Trigger: {c3_vol_trigger_strike_5dte:.0f}")
else:
    print("   • Vol Trigger not found (5 DTE)")

if c3_gamma_flip_strike_5dte is not None:
    print(f"   • Gamma Flip: {c3_gamma_flip_strike_5dte:.0f}")
else:
    print("   • Gamma Flip not found (5 DTE)")


print("\n💎 TOP 3 GEX LEVELS (5 DTE):")
if not gex_levels_5dte.empty:
    for i in range(min(3, len(gex_levels_5dte))):
        data = gex_levels_5dte.iloc[i]
        net_gex = (data['CallGEX'] + data['PutGEX']) / 10**9
        print(f"   {i+1}. Strike {data.name:.0f}: Net GEX = {net_gex:.2f} Bn")
else:
    print("   • No data for Top 3 GEX Levels (5 DTE)")


print("\n📈 STATUS DO MERCADO (5 DTE):")
print(f"   • Spot Price: ${spot_price:,.2f}")
print(f"   • Total Gamma (5 DTE): ${total_gamma_5dte:.2f} Bn")

regime_5dte = "POSITIVE GAMMA ✅" if spot_price > (c3_gamma_flip_strike_5dte if c3_gamma_flip_strike_5dte is not None else float('inf')) else "NEGATIVE GAMMA ⚠️"
print(f"   • Regime (5 DTE): {regime_5dte}")

if c3_gamma_flip_strike_5dte is not None and c3_gamma_flip_strike_5dte != 0:
    distance_to_flip_5dte = ((spot_price - c3_gamma_flip_strike_5dte) / c3_gamma_flip_strike_5dte) * 100
    print(f"   • Distância do Flip (5 DTE): {distance_to_flip_5dte:.2f}%")
else:
     print("   • Distância do Flip (5 DTE): N/A (Gamma Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("💡 PRÓXIMOS PASSOS:")
print("="*80)
print("1. ✅ Copie os valores acima")
print("2. ✅ Abra o indicador no TradingView")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Cole os valores nos inputs correspondentes")
print("5. ✅ Clique em 'OK' para aplicar")
print("\n💾 Dica: Salve este código em um arquivo .txt para referência futura!")
print("="*80 + "\n")


# ==================== VALIDAÇÃO FINAL (5 DTE) ====================

print("🔍 VALIDAÇÃO FINAL (5 DTE):\n")

# Verifica se há dados válidos
errors_final_5dte = []

if spotPrice <= 0:
    errors_final_5dte.append("❌ Spot Price inválido")

if smallest_gamma_5dte.empty and largest_gamma_5dte.empty and gex_levels_5dte.empty and (c3_gamma_flip_strike_5dte is None or c3_vol_trigger_strike_5dte is None):
    errors_final_5dte.append("❌ Nenhum dado válido encontrado para 5 DTE.")


if errors_final_5dte:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors_final_5dte:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso (5 DTE)!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (5 DTE - VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (5 DTE):

6894.6499,81.70765918278646,28 Oct 2025 00:00,-4.35693015835941,6840.0,-3.3469666044830726,6835.0,-0.44685926856845554,6830.0,21.94356905280203,6895.0,19.871875026251104,6900.0,6.852547238308764,6905.0,4.440892098500626e-14,6811.206552435575,6895.0,22.237751563063586,-0.2941825102615574,6900.0,21.02840693708854,-1.1565319108374368,6905.0,7.0102010364670635,-0.15765379815829994,6840.0,0.9767547664819792,-5.333684924841389,6850.0,4.131793974594137,-1.971744418226592,6890.0,4.665230919202307,-0.37760378280429846,6.14626876500589,6824.534816271186,60.98075016174951,6918.021594576271,-30.091773551419404,6590.817870508475

💡 COMO USAR (5 DTE):

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados d

In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK (0 DTE) - CORRIGIDA ====================
# Este célula consolida os resultados filtrados para 0 DTE e gera o código TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (0 DTE - VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (0 DTE) ====================

# CHART 1 - Spot Gamma Levels (0 DTE)
# Need to check if the smallest_gamma_0dte and largest_gamma_0dte DataFrames are not empty
if not smallest_gamma_0dte.empty:
    c1_put_wall_value_0dte = smallest_gamma_0dte.iloc[0]['TotalGamma']
    c1_put_wall_strike_0dte = smallest_gamma_0dte.index[0]
    c1_lg1_value_0dte = smallest_gamma_0dte.iloc[1]['TotalGamma']
    c1_lg1_strike_0dte = smallest_gamma_0dte.index[1]
    c1_lg2_value_0dte = smallest_gamma_0dte.iloc[2]['TotalGamma']
    c1_lg2_strike_0dte = smallest_gamma_0dte.index[2]
else:
    # Assign default values if empty
    c1_put_wall_value_0dte = 0.0
    c1_put_wall_strike_0dte = 0
    c1_lg1_value_0dte = 0.0
    c1_lg1_strike_0dte = 0
    c1_lg2_value_0dte = 0.0
    c1_lg2_strike_0dte = 0

if not largest_gamma_0dte.empty:
    c1_call_wall_value_0dte = largest_gamma_0dte.iloc[2]['TotalGamma']
    c1_call_wall_strike_0dte = largest_gamma_0dte.index[2]
    c1_lg3_value_0dte = largest_gamma_0dte.iloc[1]['TotalGamma']
    c1_lg3_strike_0dte = largest_gamma_0dte.index[1]
    c1_lg4_value_0dte = largest_gamma_0dte.iloc[0]['TotalGamma']
    c1_lg4_strike_0dte = largest_gamma_0dte.index[0]
else:
    # Assign default values if empty
    c1_call_wall_value_0dte = 0.0
    c1_call_wall_strike_0dte = 0
    c1_lg3_value_0dte = 0.0
    c1_lg3_strike_0dte = 0
    c1_lg4_value_0dte = 0.0
    c1_lg4_strike_0dte = 0


# Use the interpolated Vol Trigger from Chart 3 data for 0 DTE
c3_vol_trigger_value_0dte = vol_trigger_value_at_flip_0dte # Use the variable from the Chart 3 data cell for 0 DTE
c3_vol_trigger_strike_0dte = zeroGamma_0dte # Use the zero gamma cross strike for 0 DTE


# CHART 2 - GEX Levels (0 DTE)
# Need to check if gex_levels_0dte DataFrame is not empty
if not gex_levels_0dte.empty:
    c2_gex1_strike_0dte = gex_levels_0dte.index[0]
    c2_gex1_call_0dte = gex_levels_0dte.iloc[0]['CallGEX'] / 10**9
    c2_gex1_put_0dte = gex_levels_0dte.iloc[0]['PutGEX'] / 10**9

    c2_gex2_strike_0dte = gex_levels_0dte.index[1]
    c2_gex2_call_0dte = gex_levels_0dte.iloc[1]['CallGEX'] / 10**9
    c2_gex2_put_0dte = gex_levels_0dte.iloc[1]['PutGEX'] / 10**9

    c2_gex3_strike_0dte = gex_levels_0dte.index[2]
    c2_gex3_call_0dte = gex_levels_0dte.iloc[2]['CallGEX'] / 10**9
    c2_gex3_put_0dte = gex_levels_0dte.iloc[2]['PutGEX'] / 10**9

    # Handle cases where there might be fewer than 6 GEX levels
    c2_gex4_strike_0dte = gex_levels_0dte.index[3] if len(gex_levels_0dte) > 3 else 0
    c2_gex4_call_0dte = (gex_levels_0dte.iloc[3]['CallGEX'] / 10**9) if len(gex_levels_0dte) > 3 else 0.0
    c2_gex4_put_0dte = (gex_levels_0dte.iloc[3]['PutGEX'] / 10**9) if len(gex_levels_0dte) > 3 else 0.0

    c2_gex5_strike_0dte = gex_levels_0dte.index[4] if len(gex_levels_0dte) > 4 else 0
    c2_gex5_call_0dte = (gex_levels_0dte.iloc[4]['CallGEX'] / 10**9) if len(gex_levels_0dte) > 4 else 0.0
    c2_gex5_put_0dte = (gex_levels_0dte.iloc[4]['PutGEX'] / 10**9) if len(gex_levels_0dte) > 4 else 0.0

    c2_gex6_strike_0dte = gex_levels_0dte.index[5] if len(gex_levels_0dte) > 5 else 0
    c2_gex6_call_0dte = (gex_levels_0dte.iloc[5]['CallGEX'] / 10**9) if len(gex_levels_0dte) > 5 else 0.0
    c2_gex6_put_0dte = (gex_levels_0dte.iloc[5]['PutGEX'] / 10**9) if len(gex_levels_0dte) > 5 else 0.0
else:
    # Assign default values if empty
    c2_gex1_strike_0dte = 0
    c2_gex1_call_0dte = 0.0
    c2_gex1_put_0dte = 0.0
    c2_gex2_strike_0dte = 0
    c2_gex2_call_0dte = 0.0
    c2_gex2_put_0dte = 0.0
    c2_gex3_strike_0dte = 0
    c2_gex3_call_0dte = 0.0
    c2_gex3_put_0dte = 0.0
    c2_gex4_strike_0dte = 0
    c2_gex4_call_0dte = 0.0
    c2_gex4_put_0dte = 0.0
    c2_gex5_strike_0dte = 0
    c2_gex5_call_0dte = 0.0
    c2_gex5_put_0dte = 0.0
    c2_gex6_strike_0dte = 0
    c2_gex6_call_0dte = 0.0
    c2_gex6_put_0dte = 0.0



# CHART 3 - Gamma Profile (0 DTE)
c3_gamma_flip_value_0dte = gamma_flip_value_0dte if gamma_flip_value_0dte is not None else 0.0
c3_gamma_flip_strike_0dte = gamma_flip_strike_0dte if gamma_flip_strike_0dte is not None else 0

c3_max_pos_value_0dte = max_gamma_positive_value_0dte if 'max_gamma_positive_value_0dte' in locals() else 0.0
c3_max_pos_strike_0dte = max_gamma_positive_strike_0dte if 'max_gamma_positive_strike_0dte' in locals() else 0

c3_min_neg_value_0dte = min_gamma_negative_value_0dte if 'min_gamma_negative_value_0dte' in locals() else 0.0
c3_min_neg_strike_0dte = min_gamma_negative_strike_0dte if 'min_gamma_negative_strike_0dte' in locals() else 0


# Dados gerais (using 0 DTE total gamma)
spot_price = spotPrice # Spot price remains the same
total_gamma_0dte = df_0dte['TotalGamma'].sum() # Use total gamma for 0 DTE
update_date = todayDate.strftime('%d %b %Y 00:00') # Date remains the same

# ==================== GERAÇÃO DA LINHA ÚNICA (0 DTE) ====================

# Criar a string com todos os dados separados por vírgula
data_string_0dte = f"{spot_price},{total_gamma_0dte},{update_date},"
data_string_0dte += f"{c1_put_wall_value_0dte},{c1_put_wall_strike_0dte},"
data_string_0dte += f"{c1_lg1_value_0dte},{c1_lg1_strike_0dte},"
data_string_0dte += f"{c1_lg2_value_0dte},{c1_lg2_strike_0dte},"
data_string_0dte += f"{c1_call_wall_value_0dte},{c1_call_wall_strike_0dte},"
data_string_0dte += f"{c1_lg3_value_0dte},{c1_lg3_strike_0dte},"
data_string_0dte += f"{c1_lg4_value_0dte},{c1_lg4_strike_0dte},"
data_string_0dte += f"{c3_vol_trigger_value_0dte},{c3_vol_trigger_strike_0dte}," # Use Chart 3 Vol Trigger (0 DTE)
data_string_0dte += f"{c2_gex1_strike_0dte},{c2_gex1_call_0dte},{c2_gex1_put_0dte},"
data_string_0dte += f"{c2_gex2_strike_0dte},{c2_gex2_call_0dte},{c2_gex2_put_0dte},"
data_string_0dte += f"{c2_gex3_strike_0dte},{c2_gex3_call_0dte},{c2_gex3_put_0dte},"
data_string_0dte += f"{c2_gex4_strike_0dte},{c2_gex4_call_0dte},{c2_gex4_put_0dte},"
data_string_0dte += f"{c2_gex5_strike_0dte},{c2_gex5_call_0dte},{c2_gex5_put_0dte},"
data_string_0dte += f"{c2_gex6_strike_0dte},{c2_gex6_call_0dte},{c2_gex6_put_0dte},"
data_string_0dte += f"{c3_gamma_flip_value_0dte},{c3_gamma_flip_strike_0dte},"
data_string_0dte += f"{c3_max_pos_value_0dte},{c3_max_pos_strike_0dte},"
data_string_0dte += f"{c3_min_neg_value_0dte},{c3_min_neg_strike_0dte}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (0 DTE):\n")
print("="*80)
print(data_string_0dte)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR (0 DTE):\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (0 DTE):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA (0 DTE): ${total_gamma_0dte:.2f} Bn")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Levels - 0 DTE):")
if not smallest_gamma_0dte.empty:
    print(f"   • Put Wall: {smallest_gamma_0dte.index[0]:.0f} (γ: {smallest_gamma_0dte.iloc[0]['TotalGamma']:.2f})")
    print(f"   • Large 1: {smallest_gamma_0dte.index[1]:.0f} (γ: {smallest_gamma_0dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 2: {smallest_gamma_0dte.index[2]:.0f} (γ: {smallest_gamma_0dte.iloc[2]['TotalGamma']:.2f})")
else:
    print("   • No data for Put Levels (0 DTE)")

print("\n🟢 RESISTÊNCIAS (Call Levels - 0 DTE):")
if not largest_gamma_0dte.empty:
    print(f"   • Call Wall: {largest_gamma_0dte.index[2]:.0f} (γ: {largest_gamma_0dte.iloc[2]['TotalGamma']:.2f})")
    print(f"   • Large 3: {largest_gamma_0dte.index[1]:.0f} (γ: {largest_gamma_0dte.iloc[1]['TotalGamma']:.2f})")
    print(f"   • Large 4: {largest_gamma_0dte.index[0]:.0f} (γ: {largest_gamma_0dte.iloc[0]['TotalGamma']:.2f})")
else:
    print("   • No data for Call Levels (0 DTE)")


print("\n🟠 NÍVEIS ESPECIAIS (0 DTE):")
if c3_vol_trigger_strike_0dte is not None:
    print(f"   • Vol Trigger: {c3_vol_trigger_strike_0dte:.0f}") # Use Chart 3 Vol Trigger strike (0 DTE)
else:
    print("   • Vol Trigger not found (0 DTE)")

if c3_gamma_flip_strike_0dte is not None:
    print(f"   • Gamma Flip: {c3_gamma_flip_strike_0dte:.0f}")
else:
    print("   • Gamma Flip not found (0 DTE)")


print("\n💎 TOP 3 GEX LEVELS (0 DTE):")
if not gex_levels_0dte.empty:
    for i in range(min(3, len(gex_levels_0dte))):
        data = gex_levels_0dte.iloc[i]
        net_gex = (data['CallGEX'] + data['PutGEX']) / 10**9
        print(f"   {i+1}. Strike {data.name:.0f}: Net GEX = {net_gex:.2f} Bn")
else:
    print("   • No data for Top 3 GEX Levels (0 DTE)")


print("\n📈 STATUS DO MERCADO (0 DTE):")
print(f"   • Spot Price: ${spot_price:,.2f}")
print(f"   • Total Gamma (0 DTE): ${total_gamma_0dte:.2f} Bn")

regime_0dte = "POSITIVE GAMMA ✅" if spot_price > (c3_gamma_flip_strike_0dte if c3_gamma_flip_strike_0dte is not None else float('inf')) else "NEGATIVE GAMMA ⚠️"
print(f"   • Regime (0 DTE): {regime_0dte}")

if c3_gamma_flip_strike_0dte is not None and c3_gamma_flip_strike_0dte != 0:
    distance_to_flip_0dte = ((spot_price - c3_gamma_flip_strike_0dte) / c3_gamma_flip_strike_0dte) * 100
    print(f"   • Distância do Flip (0 DTE): {distance_to_flip_0dte:.2f}%")
else:
     print("   • Distância do Flip (0 DTE): N/A (Gamma Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("💡 PRÓXIMOS PASSOS:")
print("="*80)
print("1. ✅ Copie os valores acima")
print("2. ✅ Abra o indicador no TradingView")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Cole os valores nos inputs correspondentes")
print("5. ✅ Clique em 'OK' para aplicar")
print("\n💾 Dica: Salve este código em um arquivo .txt para referência futura!")
print("="*80 + "\n")


# ==================== VALIDAÇÃO FINAL (0 DTE) ====================

print("🔍 VALIDAÇÃO FINAL (0 DTE):\n")

# Verifica se há dados válidos
errors_final_0dte = []

if spotPrice <= 0:
    errors_final_0dte.append("❌ Spot Price inválido")

if smallest_gamma_0dte.empty and largest_gamma_0dte.empty and gex_levels_0dte.empty and (c3_gamma_flip_strike_0dte is None or c3_vol_trigger_strike_0dte is None):
    errors_final_0dte.append("❌ Nenhum dado válido encontrado para 0 DTE.")


if errors_final_0dte:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors_final_0dte:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso (0 DTE)!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (0 DTE - VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (0 DTE):

6894.6499,63.30074992956797,28 Oct 2025 00:00,-4.698715416540678,6840.0,-3.2631032453059663,6835.0,-0.8712909592773947,6830.0,21.728410716838184,6895.0,17.402079608644456,6900.0,6.559429538865463,6905.0,-5.43315392675936e-14,6824.75048322986,6895.0,21.881115996863425,-0.15270528002524433,6900.0,18.214168987874707,-0.8120893792302528,6905.0,6.58084459572369,-0.02141505685822829,6840.0,0.44522402338327677,-5.143939439923955,6890.0,4.036550449796922,-0.23893594182508038,6910.0,4.087399919988368,-0.02590722749774565,26.216791472136222,6871.278205423729,42.05622023092634,6918.021594576271,-17.656233654818585,6731.048037966102

💡 COMO USAR (0 DTE):

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Da

### RESULTADOS DELTA

In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA ====================
# Este célula consolida os resultados de Delta e gera o código TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA")
print("="*80 + "\n")

# ==================== COLETA E CÁLCULO DOS DADOS - DELTA ====================

# Recalculate aggregated data for Delta
dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalculate Delta Exposure
dfAgg_delta['CallDEX'] = dfAgg_delta['CallDelta'] * dfAgg_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_delta['PutDEX'] = dfAgg_delta['PutDelta'] * dfAgg_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_delta['TotalDelta'] = (dfAgg_delta.CallDEX + dfAgg_delta.PutDEX) / 10**6 # Converting to millions

# CHART 4 - Absolute Delta Exposure
# Find the 3 strikes with the smallest (most negative) total Delta Exposure
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)

# Find the 3 strikes with the largest (most positive) total Delta Exposure
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)

c4_put_wall_value_delta = smallest_delta.iloc[0]['TotalDelta'] if not smallest_delta.empty else 0.0
c4_put_wall_strike_delta = smallest_delta.index[0] if not smallest_delta.empty else 0
c4_lg1_value_delta = smallest_delta.iloc[1]['TotalDelta'] if len(smallest_delta) > 1 else 0.0
c4_lg1_strike_delta = smallest_delta.index[1] if len(smallest_delta) > 1 else 0
c4_lg2_value_delta = smallest_delta.iloc[2]['TotalDelta'] if len(smallest_delta) > 2 else 0.0
c4_lg2_strike_delta = smallest_delta.index[2] if len(smallest_delta) > 2 else 0

c4_call_wall_value_delta = largest_delta.iloc[2]['TotalDelta'] if len(largest_delta) > 2 else 0.0
c4_call_wall_strike_delta = largest_delta.index[2] if len(largest_delta) > 2 else 0
c4_lg3_value_delta = largest_delta.iloc[1]['TotalDelta'] if len(largest_delta) > 1 else 0.0
c4_lg3_strike_delta = largest_delta.index[1] if len(largest_delta) > 1 else 0
c4_lg4_value_delta = largest_delta.iloc[0]['TotalDelta'] if not largest_delta.empty else 0.0
c4_lg4_strike_delta = largest_delta.index[0] if not largest_delta.empty else 0


# CHART 5 - Absolute Delta Exposure by Calls and Puts (DEX Levels)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX
dex_levels_delta = dfAgg_delta_sorted_dex.head(6)

# Add checks for empty DataFrame and sufficient rows
c5_dex1_strike_delta = dex_levels_delta.index[0] if not dex_levels_delta.empty else 0
c5_dex1_call_delta = (dex_levels_delta.iloc[0]['CallDEX'] / 10**6) if not dex_levels_delta.empty else 0.0
c5_dex1_put_delta = (dex_levels_delta.iloc[0]['PutDEX'] / 10**6) if not dex_levels_delta.empty else 0.0

c5_dex2_strike_delta = dex_levels_delta.index[1] if len(dex_levels_delta) > 1 else 0
c5_dex2_call_delta = (dex_levels_delta.iloc[1]['CallDEX'] / 10**6) if len(dex_levels_delta) > 1 else 0.0
c5_dex2_put_delta = (dex_levels_delta.iloc[1]['PutDEX'] / 10**6) if len(dex_levels_delta) > 1 else 0.0

c5_dex3_strike_delta = dex_levels_delta.index[2] if len(dex_levels_delta) > 2 else 0
c5_dex3_call_delta = (dex_levels_delta.iloc[2]['CallDEX'] / 10**6) if len(dex_levels_delta) > 2 else 0.0
c5_dex3_put_delta = (dex_levels_delta.iloc[2]['PutDEX'] / 10**6) if len(dex_levels_delta) > 2 else 0.0

c5_dex4_strike_delta = dex_levels_delta.index[3] if len(dex_levels_delta) > 3 else 0
c5_dex4_call_delta = (dex_levels_delta.iloc[3]['CallDEX'] / 10**6) if len(dex_levels_delta) > 3 else 0.0
c5_dex4_put_delta = (dex_levels_delta.iloc[3]['PutDEX'] / 10**6) if len(dex_levels_delta) > 3 else 0.0

c5_dex5_strike_delta = dex_levels_delta.index[4] if len(dex_levels_delta) > 4 else 0
c5_dex5_call_delta = (dex_levels_delta.iloc[4]['CallDEX'] / 10**6) if len(dex_levels_delta) > 4 else 0.0
c5_dex5_put_delta = (dex_levels_delta.iloc[4]['PutDEX'] / 10**6) if len(dex_levels_delta) > 4 else 0.0

c5_dex6_strike_delta = dex_levels_delta.index[5] if len(dex_levels_delta) > 5 else 0
c5_dex6_call_delta = (dex_levels_delta.iloc[5]['CallDEX'] / 10**6) if len(dex_levels_delta) > 5 else 0.0
c5_dex6_put_delta = (dex_levels_delta.iloc[5]['PutDEX'] / 10**6) if len(dex_levels_delta) > 5 else 0.0


# CHART 6 - Delta Exposure Profile
# Need to recalculate the delta profile here as well
levels_delta = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalDelta = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions


# Delta Flip is the interpolated value of TotalDelta at the spotPrice
# The strike for the Delta Flip is the spot price itself
c6_delta_flip_value_delta = np.interp(spotPrice, levels_delta, totalDelta) if len(levels_delta) > 1 and len(totalDelta) > 1 else 0.0
c6_delta_flip_strike_delta = spotPrice


# Max Positive Delta (All Expiries)
c6_max_pos_value_delta = np.max(totalDelta) if len(totalDelta) > 0 else 0.0
c6_max_pos_strike_delta = levels_delta[np.argmax(totalDelta)] if len(totalDelta) > 0 else 0

# Min Negative Delta (All Expiries)
c6_min_neg_value_delta = np.min(totalDelta) if len(totalDelta) > 0 else 0.0
c6_min_neg_strike_delta = levels_delta[np.argmin(totalDelta)] if len(totalDelta) > 0 else 0


# Dados gerais (using total delta for all expiries)
spot_price = spotPrice
total_delta = df['TotalDelta'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA - DELTA ====================

# Criar a string com todos os dados separados por vírgula
data_string_delta = f"{spot_price},{total_delta},{update_date},"
data_string_delta += f"{c4_put_wall_value_delta},{c4_put_wall_strike_delta},"
data_string_delta += f"{c4_lg1_value_delta},{c4_lg1_strike_delta},"
data_string_delta += f"{c4_lg2_value_delta},{c4_lg2_strike_delta},"
data_string_delta += f"{c4_call_wall_value_delta},{c4_call_wall_strike_delta},"
data_string_delta += f"{c4_lg3_value_delta},{c4_lg3_strike_delta},"
data_string_delta += f"{c4_lg4_value_delta},{c4_lg4_strike_delta},"
data_string_delta += f"{c6_delta_flip_value_delta},{c6_delta_flip_strike_delta},"
data_string_delta += f"{c5_dex1_strike_delta},{c5_dex1_call_delta},{c5_dex1_put_delta},"
data_string_delta += f"{c5_dex2_strike_delta},{c5_dex2_call_delta},{c5_dex2_put_delta},"
data_string_delta += f"{c5_dex3_strike_delta},{c5_dex3_call_delta},{c5_dex3_put_delta},"
data_string_delta += f"{c5_dex4_strike_delta},{c5_dex4_call_delta},{c5_dex4_put_delta},"
data_string_delta += f"{c5_dex5_strike_delta},{c5_dex5_call_delta},{c5_dex5_put_delta},"
data_string_delta += f"{c5_dex6_strike_delta},{c5_dex6_call_delta},{c5_dex6_put_delta},"
data_string_delta += f"{c6_max_pos_value_delta},{c6_max_pos_strike_delta},"
data_string_delta += f"{c6_min_neg_value_delta},{c6_min_neg_strike_delta}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA):\n")
print("="*80)
print(data_string_delta)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR (DELTA):\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador correspondente para Delta")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (DELTA):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL DELTA: ${total_delta:,.2f} Million per 1% SPX Move")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Delta Levels):")
if not smallest_delta.empty:
    print(f"   • Put Wall: {smallest_delta.index[0]:.0f} (Δ: {smallest_delta.iloc[0]['TotalDelta']:.2f})")
    if len(smallest_delta) > 1:
        print(f"   • Large 1: {smallest_delta.index[1]:.0f} (Δ: {smallest_delta.iloc[1]['TotalDelta']:.2f})")
    if len(smallest_delta) > 2:
        print(f"   • Large 2: {smallest_delta.index[2]:.0f} (Δ: {smallest_delta.iloc[2]['TotalDelta']:.2f})")
else:
    print("   • No data for Put Delta Levels")

print("\n🟢 RESISTÊNCIAS (Call Delta Levels):")
if not largest_delta.empty:
    if len(largest_delta) > 2:
        print(f"   • Call Wall: {largest_delta.index[2]:.0f} (Δ: {largest_delta.iloc[2]['TotalDelta']:.2f})")
    if len(largest_delta) > 1:
        print(f"   • Large 3: {largest_delta.index[1]:.0f} (Δ: {largest_delta.iloc[1]['TotalDelta']:.2f})")
    print(f"   • Large 4: {largest_delta.index[0]:.0f} (Δ: {largest_delta.iloc[0]['TotalDelta']:.2f})")
else:
    print("   • No data for Call Delta Levels")


print("\n🟠 NÍVEIS ESPECIAIS (DELTA):")
if c6_delta_flip_strike_delta is not None:
    print(f"   • Delta Flip: {c6_delta_flip_strike_delta:.0f}")
else:
    print("   • Delta Flip not found.")

print("\n💎 TOP 3 DEX LEVELS:")
if not dex_levels_delta.empty:
    for i in range(min(3, len(dex_levels_delta))):
        data = dex_levels_delta.iloc[i]
        net_dex = (data['CallDEX'] + data['PutDEX']) / 10**6
        print(f"   {i+1}. Strike {data.name:.0f}: Net DEX = {net_dex:.2f} Million")
else:
    print("   • No data for Top 3 DEX Levels")

# Delta Regime - based on Total Delta
delta_regime = "POSITIVE DELTA ✅" if total_delta >= 0 else "NEGATIVE DELTA ⚠️"
print(f"\n📈 REGIME (DELTA): {delta_regime}")

# Distance to Delta Flip
if c6_delta_flip_strike_delta is not None and c6_delta_flip_strike_delta != 0:
    distance_to_delta_flip = ((spot_price - c6_delta_flip_strike_delta) / c6_delta_flip_strike_delta) * 100
    print(f"📏 DISTÂNCIA DO FLIP (DELTA): {distance_to_delta_flip:.2f}%")
else:
    print("📏 DISTÂNCIA DO FLIP (DELTA): N/A (Delta Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("✅ DADOS DELTA PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO - DELTA ====================

print("🔍 VALIDAÇÃO (DELTA):\n")

errors_delta = []
if spot_price <= 0:
    errors_delta.append("❌ Spot Price inválido")
if not smallest_delta.empty and smallest_delta.index[0] <= 0:
     errors_delta.append("❌ Put Delta Wall inválido")
if not largest_delta.empty and len(largest_delta) > 2 and largest_delta.index[2] <= 0:
     errors_delta.append("❌ Call Delta Wall inválido")
if c6_delta_flip_strike_delta is None or c6_delta_flip_strike_delta <= 0:
    errors_delta.append("❌ Delta Flip inválido")

if errors_delta:
    print("⚠️ AVISOS:")
    for error in errors_delta:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados (DELTA)!")
    print("✅ String pronta para uso (DELTA)!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA):

6894.6499,2925055.0094434456,28 Oct 2025 00:00,-85950.26101747093,6330.0,-73797.99138227994,6835.0,-62540.55084795688,6840.0,41276411.044062056,5000.0,32341891.03519646,6000.0,15199653.854034567,4000.0,2925055.0094434456,6894.6499,5000.0,42893786.84441107,-1617375.800349018,6000.0,36200896.81464716,-3859005.779450705,4000.0,15482683.904244337,-283030.0502097722,7000.0,5566437.990588305,-2764525.3899557455,6700.0,5517370.144938949,-1801696.843197008,6600.0,5329642.936339307,-1401205.5469432592,3510066.0113321347,8273.57988,2340044.0075547565,5515.7199200000005

💡 COMO USAR (DELTA):

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador correspondente para Delta
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'
5. ✅ COLE a linha copiada neste campo
6. ✅ Clique em 'OK'
7. ✅ PRONTO! Todos os dad

In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA (5 DTE) ====================
# Este célula consolida os resultados de Delta filtrados para 5 DTE e gera o código TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (5 DTE)")
print("="*80 + "\n")

# ==================== COLETA E CÁLCULO DOS DADOS - DELTA (5 DTE) ====================

# Filter data for 5 DTE
df_5dte_delta = df[df['daysTillExp'] <= 5/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 5 DTE
dfAgg_5dte_delta = df_5dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalculate Delta Exposure for 5 DTE
dfAgg_5dte_delta['CallDEX'] = dfAgg_5dte_delta['CallDelta'] * dfAgg_5dte_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_5dte_delta['PutDEX'] = dfAgg_5dte_delta['PutDelta'] * dfAgg_5dte_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_5dte_delta['TotalDelta'] = (dfAgg_5dte_delta.CallDEX + dfAgg_5dte_delta.PutDEX) / 10**6 # Converting to millions

# CHART 4 - Absolute Delta Exposure (5 DTE)
# Find the 3 strikes with the smallest (most negative) total Delta Exposure for 5 DTE
smallest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').head(3)

# Find the 3 strikes with the largest (most positive) total Delta Exposure for 5 DTE
largest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').tail(3)

c4_put_wall_value_delta_5dte = smallest_delta_5dte.iloc[0]['TotalDelta'] if not smallest_delta_5dte.empty else 0.0
c4_put_wall_strike_delta_5dte = smallest_delta_5dte.index[0] if not smallest_delta_5dte.empty else 0
c4_lg1_value_delta_5dte = smallest_delta_5dte.iloc[1]['TotalDelta'] if len(smallest_delta_5dte) > 1 else 0.0
c4_lg1_strike_delta_5dte = smallest_delta_5dte.index[1] if len(smallest_delta_5dte) > 1 else 0
c4_lg2_value_delta_5dte = smallest_delta_5dte.iloc[2]['TotalDelta'] if len(smallest_delta_5dte) > 2 else 0.0
c4_lg2_strike_delta_5dte = smallest_delta_5dte.index[2] if len(smallest_delta_5dte) > 2 else 0

c4_call_wall_value_delta_5dte = largest_delta_5dte.iloc[2]['TotalDelta'] if len(largest_delta_5dte) > 2 else 0.0
c4_call_wall_strike_delta_5dte = largest_delta_5dte.index[2] if len(largest_delta_5dte) > 2 else 0
c4_lg3_value_delta_5dte = largest_delta_5dte.iloc[1]['TotalDelta'] if len(largest_delta_5dte) > 1 else 0.0
c4_lg3_strike_delta_5dte = largest_delta_5dte.index[1] if len(largest_delta_5dte) > 1 else 0
c4_lg4_value_delta_5dte = largest_delta_5dte.iloc[0]['TotalDelta'] if not largest_delta_5dte.empty else 0.0
c4_lg4_strike_delta_5dte = largest_delta_5dte.index[0] if not largest_delta_5dte.empty else 0


# CHART 5 - Absolute Delta Exposure by Calls and Puts (DEX Levels - 5 DTE)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike for 5 DTE
dfAgg_5dte_delta['AbsoluteTotalDEX'] = dfAgg_5dte_delta['CallDEX'].abs() + dfAgg_5dte_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure for 5 DTE
dfAgg_delta_sorted_dex_5dte = dfAgg_5dte_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX for 5 DTE
dex_levels_delta_5dte = dfAgg_delta_sorted_dex_5dte.head(6)

# Add checks for empty DataFrame and sufficient rows
c5_dex1_strike_delta_5dte = dex_levels_delta_5dte.index[0] if not dex_levels_delta_5dte.empty else 0
c5_dex1_call_delta_5dte = (dex_levels_delta_5dte.iloc[0]['CallDEX'] / 10**6) if not dex_levels_delta_5dte.empty else 0.0
c5_dex1_put_delta_5dte = (dex_levels_delta_5dte.iloc[0]['PutDEX'] / 10**6) if not dex_levels_delta_5dte.empty else 0.0

c5_dex2_strike_delta_5dte = dex_levels_delta_5dte.index[1] if len(dex_levels_delta_5dte) > 1 else 0
c5_dex2_call_delta_5dte = (dex_levels_delta_5dte.iloc[1]['CallDEX'] / 10**6) if len(dex_levels_delta_5dte) > 1 else 0.0
c5_dex2_put_delta_5dte = (dex_levels_delta_5dte.iloc[1]['PutDEX'] / 10**6) if len(dex_levels_delta_5dte) > 1 else 0.0

c5_dex3_strike_delta_5dte = dex_levels_delta_5dte.index[2] if len(dex_levels_delta_5dte) > 2 else 0
c5_dex3_call_delta_5dte = (dex_levels_delta_5dte.iloc[2]['CallDEX'] / 10**6) if len(dex_levels_delta_5dte) > 2 else 0.0
c5_dex3_put_delta_5dte = (dex_levels_delta_5dte.iloc[2]['PutDEX'] / 10**6) if len(dex_levels_delta_5dte) > 2 else 0.0

c5_dex4_strike_delta_5dte = dex_levels_delta_5dte.index[3] if len(dex_levels_delta_5dte) > 3 else 0
c5_dex4_call_delta_5dte = (dex_levels_delta_5dte.iloc[3]['CallDEX'] / 10**6) if len(dex_levels_delta_5dte) > 3 else 0.0
c5_dex4_put_delta_5dte = (dex_levels_delta_5dte.iloc[3]['PutDEX'] / 10**6) if len(dex_levels_delta_5dte) > 3 else 0.0

c5_dex5_strike_delta_5dte = dex_levels_delta_5dte.index[4] if len(dex_levels_delta_5dte) > 4 else 0
c5_dex5_call_delta_5dte = (dex_levels_delta_5dte.iloc[4]['CallDEX'] / 10**6) if len(dex_levels_delta_5dte) > 4 else 0.0
c5_dex5_put_delta_5dte = (dex_levels_delta_5dte.iloc[4]['PutDEX'] / 10**6) if len(dex_levels_delta_5dte) > 4 else 0.0

c5_dex6_strike_delta_5dte = dex_levels_delta_5dte.index[5] if len(dex_levels_delta_5dte) > 5 else 0
c5_dex6_call_delta_5dte = (dex_levels_delta_5dte.iloc[5]['CallDEX'] / 10**6) if len(dex_levels_delta_5dte) > 5 else 0.0
c5_dex6_put_delta_5dte = (dex_levels_delta_5dte.iloc[5]['PutDEX'] / 10**6) if len(dex_levels_delta_5dte) > 5 else 0.0


# CHART 6 - Delta Exposure Profile (5 DTE)
# Need to recalculate the delta profile for 5 DTE
df_5dte_profile_delta = df[df['daysTillExp'] <= 5/262].copy() # Use a copy

levels_delta_5dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalDelta_5dte = []

# For each spot level, calc delta exposure at that point for 5 DTE data
for level in levels_delta_5dte:
    df_5dte_profile_delta['callDeltaEx'] = df_5dte_profile_delta.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df_5dte_profile_delta['putDeltaEx'] = df_5dte_profile_delta.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta_5dte.append(df_5dte_profile_delta['callDeltaEx'].sum() + df_5dte_profile_delta['putDeltaEx'].sum())

totalDelta_5dte = np.array(totalDelta_5dte) / 10**6 # Converting to millions


# Delta Flip is the interpolated value of TotalDelta at the spotPrice for 5 DTE data
# The strike for the Delta Flip is the spot price itself
c6_delta_flip_value_delta_5dte = np.interp(spotPrice, levels_delta_5dte, totalDelta_5dte) if len(levels_delta_5dte) > 1 and len(totalDelta_5dte) > 1 else 0.0
c6_delta_flip_strike_delta_5dte = spotPrice


# Max Positive Delta (5 DTE)
c6_max_pos_value_delta_5dte = np.max(totalDelta_5dte) if len(totalDelta_5dte) > 0 else 0.0
c6_max_pos_strike_delta_5dte = levels_delta_5dte[np.argmax(totalDelta_5dte)] if len(totalDelta_5dte) > 0 else 0

# Min Negative Delta (5 DTE)
c6_min_neg_value_delta_5dte = np.min(totalDelta_5dte) if len(totalDelta_5dte) > 0 else 0.0
c6_min_neg_strike_delta_5dte = levels_delta_5dte[np.argmin(totalDelta_5dte)] if len(totalDelta_5dte) > 0 else 0


# Dados gerais (using total delta for 5 DTE)
spot_price = spotPrice
total_delta_5dte = df_5dte_delta['TotalDelta'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA - DELTA (5 DTE) ====================

# Criar a string com todos os dados separados por vírgula
data_string_delta_5dte = f"{spot_price},{total_delta_5dte},{update_date},"
data_string_delta_5dte += f"{c4_put_wall_value_delta_5dte},{c4_put_wall_strike_delta_5dte},"
data_string_delta_5dte += f"{c4_lg1_value_delta_5dte},{c4_lg1_strike_delta_5dte},"
data_string_delta_5dte += f"{c4_lg2_value_delta_5dte},{c4_lg2_strike_delta_5dte},"
data_string_delta_5dte += f"{c4_call_wall_value_delta_5dte},{c4_call_wall_strike_delta_5dte},"
data_string_delta_5dte += f"{c4_lg3_value_delta_5dte},{c4_lg3_strike_delta_5dte},"
data_string_delta_5dte += f"{c4_lg4_value_delta_5dte},{c4_lg4_strike_delta_5dte},"
data_string_delta_5dte += f"{c6_delta_flip_value_delta_5dte},{c6_delta_flip_strike_delta_5dte},"
data_string_delta_5dte += f"{c5_dex1_strike_delta_5dte},{c5_dex1_call_delta_5dte},{c5_dex1_put_delta_5dte},"
data_string_delta_5dte += f"{c5_dex2_strike_delta_5dte},{c5_dex2_call_delta_5dte},{c5_dex2_put_delta_5dte},"
data_string_delta_5dte += f"{c5_dex3_strike_delta_5dte},{c5_dex3_call_delta_5dte},{c5_dex3_put_delta_5dte},"
data_string_delta_5dte += f"{c5_dex4_strike_delta_5dte},{c5_dex4_call_delta_5dte},{c5_dex4_put_delta_5dte},"
data_string_delta_5dte += f"{c5_dex5_strike_delta_5dte},{c5_dex5_call_delta_5dte},{c5_dex5_put_delta_5dte},"
data_string_delta_5dte += f"{c5_dex6_strike_delta_5dte},{c5_dex6_call_delta_5dte},{c5_dex6_put_delta_5dte},"
data_string_delta_5dte += f"{c6_max_pos_value_delta_5dte},{c6_max_pos_strike_delta_5dte},"
data_string_delta_5dte += f"{c6_min_neg_value_delta_5dte},{c6_min_neg_strike_delta_5dte}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA - 5 DTE):\n")
print("="*80)
print(data_string_delta_5dte)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR (DELTA - 5 DTE):\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador correspondente para Delta")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (DELTA - 5 DTE):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL DELTA (5 DTE): ${total_delta_5dte:,.2f} Million per 1% SPX Move")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Delta Levels - 5 DTE):")
if not smallest_delta_5dte.empty:
    print(f"   • Put Wall: {smallest_delta_5dte.index[0]:.0f} (Δ: {smallest_delta_5dte.iloc[0]['TotalDelta']:.2f})")
    if len(smallest_delta_5dte) > 1:
        print(f"   • Large 1: {smallest_delta_5dte.index[1]:.0f} (Δ: {smallest_delta_5dte.iloc[1]['TotalDelta']:.2f})")
    if len(smallest_delta_5dte) > 2:
        print(f"   • Large 2: {smallest_delta_5dte.index[2]:.0f} (Δ: {smallest_delta_5dte.iloc[2]['TotalDelta']:.2f})")
else:
    print("   • No data for Put Delta Levels (5 DTE)")

print("\n🟢 RESISTÊNCIAS (Call Delta Levels - 5 DTE):")
if not largest_delta_5dte.empty:
    if len(largest_delta_5dte) > 2:
        print(f"   • Call Wall: {largest_delta_5dte.index[2]:.0f} (Δ: {largest_delta_5dte.iloc[2]['TotalDelta']:.2f})")
    if len(largest_delta_5dte) > 1:
        print(f"   • Large 3: {largest_delta_5dte.index[1]:.0f} (Δ: {largest_delta_5dte.iloc[1]['TotalDelta']:.2f})")
    print(f"   • Large 4: {largest_delta_5dte.index[0]:.0f} (Δ: {largest_delta_5dte.iloc[0]['TotalDelta']:.2f})")
else:
    print("   • No data for Call Delta Levels (5 DTE)")


print("\n🟠 NÍVEIS ESPECIAIS (DELTA - 5 DTE):")
if c6_delta_flip_strike_delta_5dte is not None:
    print(f"   • Delta Flip: {c6_delta_flip_strike_delta_5dte:.0f}")
else:
    print("   • Delta Flip not found (5 DTE).")

print("\n💎 TOP 3 DEX LEVELS (5 DTE):")
if not dex_levels_delta_5dte.empty:
    for i in range(min(3, len(dex_levels_delta_5dte))):
        data = dex_levels_delta_5dte.iloc[i]
        net_dex = (data['CallDEX'] + data['PutDEX']) / 10**6
        print(f"   {i+1}. Strike {data.name:.0f}: Net DEX = {net_dex:.2f} Million")
else:
    print("   • No data for Top 3 DEX Levels (5 DTE)")


# Delta Regime - based on Total Delta (5 DTE)
delta_regime_5dte = "POSITIVE DELTA ✅" if total_delta_5dte >= 0 else "NEGATIVE DELTA ⚠️"
print(f"\n📈 REGIME (DELTA - 5 DTE): {delta_regime_5dte}")

# Distance to Delta Flip (5 DTE)
if c6_delta_flip_strike_delta_5dte is not None and c6_delta_flip_strike_delta_5dte != 0:
    distance_to_delta_flip_5dte = ((spot_price - c6_delta_flip_strike_delta_5dte) / c6_delta_flip_strike_delta_5dte) * 100
    print(f"📏 DISTÂNCIA DO FLIP (DELTA - 5 DTE): {distance_to_delta_flip_5dte:.2f}%")
else:
    print("📏 DISTÂNCIA DO FLIP (DELTA - 5 DTE): N/A (Delta Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("✅ DADOS DELTA (5 DTE) PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO - DELTA (5 DTE) ====================

print("🔍 VALIDAÇÃO (DELTA - 5 DTE):\n")

errors_delta_5dte = []
if spot_price <= 0:
    errors_delta_5dte.append("❌ Spot Price inválido")
if not smallest_delta_5dte.empty and smallest_delta_5dte.index[0] <= 0:
     errors_delta_5dte.append("❌ Put Delta Wall (5 DTE) inválido")
if not largest_delta_5dte.empty and len(largest_delta_5dte) > 2 and largest_delta_5dte.index[2] <= 0:
     errors_delta_5dte.append("❌ Call Delta Wall (5 DTE) inválido")
if c6_delta_flip_strike_delta_5dte is None or c6_delta_flip_strike_delta_5dte <= 0:
    errors_delta_5dte.append("❌ Delta Flip (5 DTE) inválido")

if errors_delta_5dte:
    print("⚠️ AVISOS:")
    for error in errors_delta_5dte:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados (DELTA - 5 DTE)!")
    print("✅ String pronta para uso (DELTA - 5 DTE)!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (5 DTE)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA - 5 DTE):

6894.6499,182617.48386211455,28 Oct 2025 00:00,-20370.17370042551,7000.0,-15461.407668265747,6835.0,-10348.716714458214,6840.0,72573.42868359052,6900.0,66945.18780143654,6895.0,54098.472481157776,6750.0,182617.48386211458,6894.6499,6900.0,79017.68943987695,-6444.260756286445,6850.0,63829.56232078405,-11136.096212906063,6895.0,68819.41653725722,-1874.228735820687,6800.0,60026.104434281384,-8383.243906074935,6750.0,61115.345218865055,-7016.872737707281,6650.0,54769.73118284584,-3446.892517558272,219140.98063453747,8273.57988,146093.98708969165,5515.7199200000005

💡 COMO USAR (DELTA - 5 DTE):

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador correspondente para Delta
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'
5. ✅ COLE a linha copiada neste campo
6. ✅ Clique em 'OK'


In [ ]:
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA (0 DTE) ====================
# Este célula consolida os resultados de Delta filtrados para 0 DTE e gera o código TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (0 DTE)")
print("="*80 + "\n")

# ==================== COLETA E CÁLCULO DOS DADOS - DELTA (0 DTE) ====================

# Filter data for 0 DTE (using a small epsilon or checking for 0 business days)
# Since daysTillExp is calculated as business days / 262, 0 business days will be 1/262
df_0dte_delta = df[df['daysTillExp'] <= 1/262].copy() # Use a copy to avoid SettingWithCopyWarning

# Recalculate aggregated data for 0 DTE
dfAgg_0dte_delta = df_0dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalculate Delta Exposure for 0 DTE
dfAgg_0dte_delta['CallDEX'] = dfAgg_0dte_delta['CallDelta'] * dfAgg_0dte_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_0dte_delta['PutDEX'] = dfAgg_0dte_delta['PutDelta'] * dfAgg_0dte_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_0dte_delta['TotalDelta'] = (dfAgg_0dte_delta.CallDEX + dfAgg_0dte_delta.PutDEX) / 10**6 # Converting to millions


# CHART 4 - Absolute Delta Exposure (0 DTE)
# Find the 3 strikes with the smallest (most negative) total Delta Exposure for 0 DTE
smallest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').head(3)

# Find the 3 strikes with the largest (most positive) total Delta Exposure for 0 DTE
largest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').tail(3)

c4_put_wall_value_delta_0dte = smallest_delta_0dte.iloc[0]['TotalDelta'] if not smallest_delta_0dte.empty else 0.0
c4_put_wall_strike_delta_0dte = smallest_delta_0dte.index[0] if not smallest_delta_0dte.empty else 0
c4_lg1_value_delta_0dte = smallest_delta_0dte.iloc[1]['TotalDelta'] if len(smallest_delta_0dte) > 1 else 0.0
c4_lg1_strike_delta_0dte = smallest_delta_0dte.index[1] if len(smallest_delta_0dte) > 1 else 0
c4_lg2_value_delta_0dte = smallest_delta_0dte.iloc[2]['TotalDelta'] if len(smallest_delta_0dte) > 2 else 0.0
c4_lg2_strike_delta_0dte = smallest_delta_0dte.index[2] if len(smallest_delta_0dte) > 2 else 0

c4_call_wall_value_delta_0dte = largest_delta_0dte.iloc[2]['TotalDelta'] if len(largest_delta_0dte) > 2 else 0.0
c4_call_wall_strike_delta_0dte = largest_delta_0dte.index[2] if len(largest_delta_0dte) > 2 else 0
c4_lg3_value_delta_0dte = largest_delta_0dte.iloc[1]['TotalDelta'] if len(largest_delta_0dte) > 1 else 0.0
c4_lg3_strike_delta_0dte = largest_delta_0dte.index[1] if len(largest_delta_0dte) > 1 else 0
c4_lg4_value_delta_0dte = largest_delta_0dte.iloc[0]['TotalDelta'] if not largest_delta_0dte.empty else 0.0
c4_lg4_strike_delta_0dte = largest_delta_0dte.index[0] if not largest_delta_0dte.empty else 0


# CHART 5 - Absolute Delta Exposure by Calls and Puts (DEX Levels - 0 DTE)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike for 0 DTE
dfAgg_0dte_delta['AbsoluteTotalDEX'] = dfAgg_0dte_delta['CallDEX'].abs() + dfAgg_0dte_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure for 0 DTE
dfAgg_delta_sorted_dex_0dte = dfAgg_0dte_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX for 0 DTE
dex_levels_delta_0dte = dfAgg_delta_sorted_dex_0dte.head(6)

# Add checks for empty DataFrame and sufficient rows
c5_dex1_strike_delta_0dte = dex_levels_delta_0dte.index[0] if not dex_levels_delta_0dte.empty else 0
c5_dex1_call_delta_0dte = (dex_levels_delta_0dte.iloc[0]['CallDEX'] / 10**6) if not dex_levels_delta_0dte.empty else 0.0
c5_dex1_put_delta_0dte = (dex_levels_delta_0dte.iloc[0]['PutDEX'] / 10**6) if not dex_levels_delta_0dte.empty else 0.0

c5_dex2_strike_delta_0dte = dex_levels_delta_0dte.index[1] if len(dex_levels_delta_0dte) > 1 else 0
c5_dex2_call_delta_0dte = (dex_levels_delta_0dte.iloc[1]['CallDEX'] / 10**6) if len(dex_levels_delta_0dte) > 1 else 0.0
c5_dex2_put_delta_0dte = (dex_levels_delta_0dte.iloc[1]['PutDEX'] / 10**6) if len(dex_levels_delta_0dte) > 1 else 0.0

c5_dex3_strike_delta_0dte = dex_levels_delta_0dte.index[2] if len(dex_levels_delta_0dte) > 2 else 0
c5_dex3_call_delta_0dte = (dex_levels_delta_0dte.iloc[2]['CallDEX'] / 10**6) if len(dex_levels_delta_0dte) > 2 else 0.0
c5_dex3_put_delta_0dte = (dex_levels_delta_0dte.iloc[2]['PutDEX'] / 10**6) if len(dex_levels_delta_0dte) > 2 else 0.0

c5_dex4_strike_delta_0dte = dex_levels_delta_0dte.index[3] if len(dex_levels_delta_0dte) > 3 else 0
c5_dex4_call_delta_0dte = (dex_levels_delta_0dte.iloc[3]['CallDEX'] / 10**6) if len(dex_levels_delta_0dte) > 3 else 0.0
c5_dex4_put_delta_0dte = (dex_levels_delta_0dte.iloc[3]['PutDEX'] / 10**6) if len(dex_levels_delta_0dte) > 3 else 0.0

c5_dex5_strike_delta_0dte = dex_levels_delta_0dte.index[4] if len(dex_levels_delta_0dte) > 4 else 0
c5_dex5_call_delta_0dte = (dex_levels_delta_0dte.iloc[4]['CallDEX'] / 10**6) if len(dex_levels_delta_0dte) > 4 else 0.0
c5_dex5_put_delta_0dte = (dex_levels_delta_0dte.iloc[4]['PutDEX'] / 10**6) if len(dex_levels_delta_0dte) > 4 else 0.0

c5_dex6_strike_delta_0dte = dex_levels_delta_0dte.index[5] if len(dex_levels_delta_0dte) > 5 else 0
c5_dex6_call_delta_0dte = (dex_levels_delta_0dte.iloc[5]['CallDEX'] / 10**6) if len(dex_levels_delta_0dte) > 5 else 0.0
c5_dex6_put_delta_0dte = (dex_levels_delta_0dte.iloc[5]['PutDEX'] / 10**6) if len(dex_levels_delta_0dte) > 5 else 0.0


# CHART 6 - Delta Exposure Profile (0 DTE)
# Need to recalculate the delta profile for 0 DTE
df_0dte_profile_delta = df[df['daysTillExp'] <= 1/262].copy() # Use a copy

levels_delta_0dte = np.linspace(fromStrike, toStrike, 60) # Use the same levels as before

totalDelta_0dte = []

# For each spot level, calc delta exposure at that point for 0 DTE data
for level in levels_delta_0dte:
    df_0dte_profile_delta['callDeltaEx'] = df_0dte_profile_delta.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df_0dte_profile_delta['putDeltaEx'] = df_0dte_profile_delta.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta_0dte.append(df_0dte_profile_delta['callDeltaEx'].sum() + df_0dte_profile_delta['putDeltaEx'].sum())

totalDelta_0dte = np.array(totalDelta_0dte) / 10**6 # Converting to millions


# Delta Flip is the interpolated value of TotalDelta at the spotPrice for 0 DTE data
# The strike for the Delta Flip is the spot price itself
c6_delta_flip_value_delta_0dte = np.interp(spotPrice, levels_delta_0dte, totalDelta_0dte) if len(levels_delta_0dte) > 1 and len(totalDelta_0dte) > 1 else 0.0
c6_delta_flip_strike_delta_0dte = spotPrice


# Max Positive Delta (0 DTE)
c6_max_pos_value_delta_0dte = np.max(totalDelta_0dte) if len(totalDelta_0dte) > 0 else 0.0
c6_max_pos_strike_delta_0dte = levels_delta_0dte[np.argmax(totalDelta_0dte)] if len(totalDelta_0dte) > 0 else 0

# Min Negative Delta (0 DTE)
c6_min_neg_value_delta_0dte = np.min(totalDelta_0dte) if len(totalDelta_0dte) > 0 else 0.0
c6_min_neg_strike_delta_0dte = levels_delta_0dte[np.argmin(totalDelta_0dte)] if len(totalDelta_0dte) > 0 else 0


# Dados gerais (using total delta for 0 DTE)
spot_price = spotPrice
total_delta_0dte = df_0dte_delta['TotalDelta'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA - DELTA (0 DTE) ====================

# Criar a string com todos os dados separados por vírgula
data_string_delta_0dte = f"{spot_price},{total_delta_0dte},{update_date},"
data_string_delta_0dte += f"{c4_put_wall_value_delta_0dte},{c4_put_wall_strike_delta_0dte},"
data_string_delta_0dte += f"{c4_lg1_value_delta_0dte},{c4_lg1_strike_delta_0dte},"
data_string_delta_0dte += f"{c4_lg2_value_delta_0dte},{c4_lg2_strike_delta_0dte},"
data_string_delta_0dte += f"{c4_call_wall_value_delta_0dte},{c4_call_wall_strike_delta_0dte},"
data_string_delta_0dte += f"{c4_lg3_value_delta_0dte},{c4_lg3_strike_delta_0dte},"
data_string_delta_0dte += f"{c4_lg4_value_delta_0dte},{c4_lg4_strike_delta_0dte},"
data_string_delta_0dte += f"{c6_delta_flip_value_delta_0dte},{c6_delta_flip_strike_delta_0dte},"
data_string_delta_0dte += f"{c5_dex1_strike_delta_0dte},{c5_dex1_call_delta_0dte},{c5_dex1_put_delta_0dte},"
data_string_delta_0dte += f"{c5_dex2_strike_delta_0dte},{c5_dex2_call_delta_0dte},{c5_dex2_put_delta_0dte},"
data_string_delta_0dte += f"{c5_dex3_strike_delta_0dte},{c5_dex3_call_delta_0dte},{c5_dex3_put_delta_0dte},"
data_string_delta_0dte += f"{c5_dex4_strike_delta_0dte},{c5_dex4_call_delta_0dte},{c5_dex4_put_delta_0dte},"
data_string_delta_0dte += f"{c5_dex5_strike_delta_0dte},{c5_dex5_call_delta_0dte},{c5_dex5_put_delta_0dte},"
data_string_delta_0dte += f"{c5_dex6_strike_delta_0dte},{c5_dex6_call_delta_0dte},{c5_dex6_put_delta_0dte},"
data_string_delta_0dte += f"{c6_max_pos_value_delta_0dte},{c6_max_pos_strike_delta_0dte},"
data_string_delta_0dte += f"{c6_min_neg_value_delta_0dte},{c6_min_neg_strike_delta_0dte}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA - 0 DTE):\n")
print("="*80)
print(data_string_delta_0dte)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR (DELTA - 0 DTE):\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador correspondente para Delta")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (DELTA - 0 DTE):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL DELTA (0 DTE): ${total_delta_0dte:,.2f} Million per 1% SPX Move")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Delta Levels - 0 DTE):")
if not smallest_delta_0dte.empty:
    print(f"   • Put Wall: {smallest_delta_0dte.index[0]:.0f} (Δ: {smallest_delta_0dte.iloc[0]['TotalDelta']:.2f})")
    if len(smallest_delta_0dte) > 1:
        print(f"   • Large 1: {smallest_delta_0dte.index[1]:.0f} (Δ: {smallest_delta_0dte.iloc[1]['TotalDelta']:.2f})")
    if len(smallest_delta_0dte) > 2:
        print(f"   • Large 2: {smallest_delta_0dte.index[2]:.0f} (Δ: {smallest_delta_0dte.iloc[2]['TotalDelta']:.2f})")
else:
    print("   • No data for Put Delta Levels (0 DTE)")

print("\n🟢 RESISTÊNCIAS (Call Delta Levels - 0 DTE):")
if not largest_delta_0dte.empty:
    if len(largest_delta_0dte) > 2:
        print(f"   • Call Wall: {largest_delta_0dte.index[2]:.0f} (Δ: {largest_delta_0dte.iloc[2]['TotalDelta']:.2f})")
    if len(largest_delta_0dte) > 1:
        print(f"   • Large 3: {largest_delta_0dte.index[1]:.0f} (Δ: {largest_delta_0dte.iloc[1]['TotalDelta']:.2f})")
    print(f"   • Large 4: {largest_delta_0dte.index[0]:.0f} (Δ: {largest_delta_0dte.iloc[0]['TotalDelta']:.2f})")
else:
    print("   • No data for Call Delta Levels (0 DTE)")


print("\n🟠 NÍVEIS ESPECIAIS (DELTA - 0 DTE):")
if c6_delta_flip_strike_delta_0dte is not None:
    print(f"   • Delta Flip: {c6_delta_flip_strike_delta_0dte:.0f}")
else:
    print("   • Delta Flip not found (0 DTE).")


print("\n💎 TOP 3 DEX LEVELS (0 DTE):")
if not dex_levels_delta_0dte.empty:
    for i in range(min(3, len(dex_levels_delta_0dte))):
        data = dex_levels_delta_0dte.iloc[i]
        net_dex = (data['CallDEX'] + data['PutDEX']) / 10**6
        print(f"   {i+1}. Strike {data.name:.0f}: Net DEX = {net_dex:.2f} Million")
else:
    print("   • No data for Top 3 DEX Levels (0 DTE)")


# Delta Regime - based on Total Delta (0 DTE)
delta_regime_0dte = "POSITIVE DELTA ✅" if total_delta_0dte >= 0 else "NEGATIVE DELTA ⚠️"
print(f"\n📈 REGIME (DELTA - 0 DTE): {delta_regime_0dte}")

# Distance to Delta Flip (0 DTE)
if c6_delta_flip_strike_delta_0dte is not None and c6_delta_flip_strike_delta_0dte != 0:
    distance_to_delta_flip_0dte = ((spot_price - c6_delta_flip_strike_delta_0dte) / c6_delta_flip_strike_delta_0dte) * 100
    print(f"📏 DISTÂNCIA DO FLIP (DELTA - 0 DTE): {distance_to_delta_flip_0dte:.2f}%")
else:
    print("📏 DISTÂNCIA DO FLIP (DELTA - 0 DTE): N/A (Delta Flip not found or is 0)")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("✅ DADOS DELTA (0 DTE) PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO - DELTA (0 DTE) ====================

print("🔍 VALIDAÇÃO (DELTA - 0 DTE):\n")

errors_delta_0dte = []
if spot_price <= 0:
    errors_delta_0dte.append("❌ Spot Price inválido")
if not smallest_delta_0dte.empty and smallest_delta_0dte.index[0] <= 0:
     errors_delta_0dte.append("❌ Put Delta Wall (0 DTE) inválido")
if not largest_delta_0dte.empty and len(largest_delta_0dte) > 2 and largest_delta_0dte.index[2] <= 0:
     errors_delta_0dte.append("❌ Call Delta Wall (0 DTE) inválido")
if c6_delta_flip_strike_delta_0dte is None or c6_delta_flip_strike_delta_0dte <= 0:
    errors_delta_0dte.append("❌ Delta Flip (0 DTE) inválido")

if errors_delta_0dte:
    print("⚠️ AVISOS:")
    for error in errors_delta_0dte:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados (DELTA - 0 DTE)!")
    print("✅ String pronta para uso (DELTA - 0 DTE)!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (0 DTE)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW (DELTA - 0 DTE):

6894.6499,73162.56472769419,28 Oct 2025 00:00,-2518.9432422332484,6840.0,-2469.3663658228115,6835.0,-25.453392714824005,7175.0,20439.717482946347,6895.0,14164.579041381952,6900.0,8122.758792919009,6800.0,73162.56472769419,6894.6499,6895.0,20596.075111790036,-156.357628843689,6900.0,15144.159178269123,-979.5801368871702,6850.0,8164.630277547704,-1150.738510671189,6800.0,8711.047070870976,-588.288277951967,6840.0,2576.757915290704,-5095.701157523952,6875.0,5010.20281040202,-667.7984147962521,87795.077673233,8273.57988,58530.05178215535,5515.7199200000005

💡 COMO USAR (DELTA - 0 DTE):

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador correspondente para Delta
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'
5. ✅ COLE a linha copiada neste campo
6. ✅ Clique em 'OK'
7. ✅ PRO